#### Phase 7.1 — Cloudflare R2 Connectivity Test

In [1]:
import sys
import subprocess
import importlib.util

# Install boto3 only if missing
if importlib.util.find_spec("boto3") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "boto3"]
    )
import boto3
from google.colab import userdata
from botocore.exceptions import ClientError

# Load corrected credentials
ACCOUNT_ID = userdata.get("R2_ACCOUNT_ID").strip()
ACCESS_KEY = userdata.get("R2_ACCESS_KEY").strip()
SECRET_KEY = userdata.get("R2_SECRET_ACCESS_KEY").strip()
BUCKET_NAME = userdata.get("R2_BUCKET-NAME").strip()

# Validate access key format
assert len(ACCESS_KEY) == 32
assert len(SECRET_KEY) > 0

# Create R2 S3 client
s3_client = boto3.client(
    "s3",
    endpoint_url=f"https://{ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name="auto"
)

# Read-only bucket access test
try:
    response = s3_client.list_objects_v2(
        Bucket=BUCKET_NAME,
        MaxKeys=1
    )

    print("=" * 45)
    print("CLOUDFLARE R2 CONNECTION: SUCCESS")
    print("=" * 45)
    print("Bucket:", BUCKET_NAME)
    print("Authentication: PASS")
    print("Bucket access: PASS")
    print("Object listing: PASS")
    print("HTTP Status:",
          response["ResponseMetadata"]["HTTPStatusCode"])
    print("Objects returned:",
          len(response.get("Contents", [])))

except ClientError as e:
    print("R2 CONNECTION FAILED")
    print("Error:", e.response.get("Error", {}).get("Code"))
    print("Message:", e.response.get("Error", {}).get("Message"))

except Exception as e:
    print("ERROR:", type(e).__name__, str(e))


CLOUDFLARE R2 CONNECTION: SUCCESS
Bucket: sales-analytics-pipeline
Authentication: PASS
Bucket access: PASS
Object listing: PASS
HTTP Status: 200
Objects returned: 1


In [2]:
#Varify project root

from pathlib import Path
from google.colab import drive
import os

#mount google drive
drive.mount('/content/drive')

#set project root

REPO_URL = (
    "https://github.com/vipulkumawat17/"
    "Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data"
)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline"
)

print("PROJECT ROOT:", PROJECT_ROOT)
print("ROOT EXISTS:", PROJECT_ROOT.exists())

# Identify the actual repository location
candidates = [
    PROJECT_ROOT,
    PROJECT_ROOT  / "data"
]

for path in candidates:
    print(f"\nChecking: {path}")
    print("Exists:", path.exists())

    if path.exists():
        print("Git repository:", (path / ".git").exists())
        print("Phase 4:", (path / "data" / "processed").exists())
        print("Reports:", (path / "reports").exists())

# Inspect existing data directory

DATA_ROOT = PROJECT_ROOT / "data"

print("DATA DIRECTORY:", DATA_ROOT)
print("\nExisting folders and files:\n")

for item in sorted(DATA_ROOT.iterdir()):
    if item.is_dir():
        print(f"[DIR]  {item.name}/")
    else:
        print(f"[FILE] {item.name}")


Mounted at /content/drive
PROJECT ROOT: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline
ROOT EXISTS: True

Checking: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline
Exists: True
Git repository: True
Phase 4: True
Reports: True

Checking: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data
Exists: True
Git repository: False
Phase 4: False
Reports: False
DATA DIRECTORY: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data

Existing folders and files:

[FILE] .gitkeep
[DIR]  bronze/
[DIR]  gold/
[DIR]  processed/
[DIR]  raw/
[DIR]  rejected/
[DIR]  silver/


In [ ]:

# Inspect Phase 4 Storage Structure


# Phase 7 — Step 7.1.5
# Inspect existing Bronze, Silver and Gold layers

DATA_ROOT = PROJECT_ROOT / "data"

for layer in ["bronze", "silver", "gold"]:
    layer_path = DATA_ROOT / layer

    print(f"\n{'=' * 50}")
    print(f"{layer.upper()} STORAGE")
    print(f"{'=' * 50}")

    if not layer_path.exists():
        print("Directory missing")
        continue

    for root, dirs, files in os.walk(layer_path):
        relative_path = Path(root).relative_to(layer_path)

        depth = len(relative_path.parts)

        if depth > 2:
            dirs.clear()
            continue

        indent = "  " * depth

        print(f"{indent}{Path(root).name}/")

        for filename in sorted(files):
            file_path = Path(root) / filename
            size_mb = file_path.stat().st_size / (1024 * 1024)

            print(
                f"{indent}  {filename} "
                f"({size_mb:.2f} MB)"
            )




BRONZE STORAGE
bronze/
  olist/
    olist_customers_dataset/
      ._SUCCESS.crc (0.00 MB)
      .part-00000-6fb03181-5708-4362-bacf-c830c878fc80-c000.snappy.parquet.crc (0.04 MB)
      .part-00001-6fb03181-5708-4362-bacf-c830c878fc80-c000.snappy.parquet.crc (0.01 MB)
      _SUCCESS (0.00 MB)
      part-00000-6fb03181-5708-4362-bacf-c830c878fc80-c000.snappy.parquet (4.88 MB)
      part-00001-6fb03181-5708-4362-bacf-c830c878fc80-c000.snappy.parquet (1.82 MB)
    olist_geolocation_dataset/
      ._SUCCESS.crc (0.00 MB)
      .part-00000-070cfc71-c132-4737-9309-be6c24464ebc-c000.snappy.parquet.crc (0.00 MB)
      .part-00001-070cfc71-c132-4737-9309-be6c24464ebc-c000.snappy.parquet.crc (0.00 MB)
      _SUCCESS (0.00 MB)
      part-00000-070cfc71-c132-4737-9309-be6c24464ebc-c000.snappy.parquet (0.00 MB)
      part-00001-070cfc71-c132-4737-9309-be6c24464ebc-c000.snappy.parquet (0.00 MB)
    olist_order_items_dataset/
      ._SUCCESS.crc (0.00 MB)
      .part-00000-1e3155a0-ffdf-45ff-ace4-3b

In [ ]:

# Build R2 Migration Manifest
# Read-only: No uploads or local file modifications.

from pathlib import Path
import pandas as pd

DATA_ROOT = PROJECT_ROOT / "data"

migration_records = []

for layer in ["bronze", "silver", "gold"]:

    layer_path = DATA_ROOT / layer

    if not layer_path.exists():
        continue

    for file_path in sorted(layer_path.rglob("*.parquet")):

        relative_path = file_path.relative_to(DATA_ROOT)

        migration_records.append({
            "layer": layer,
            "dataset": file_path.parent.name,
            "filename": file_path.name,
            "local_path": str(file_path),
            "r2_object_key": relative_path.as_posix(),
            "size_bytes": file_path.stat().st_size,
            "size_mb": round(
                file_path.stat().st_size / (1024 * 1024), 4
            )
        })

migration_manifest = pd.DataFrame(migration_records)

if migration_manifest.empty:
    raise ValueError("No Parquet files found.")

print("=" * 60)
print("R2 MIGRATION MANIFEST")
print("=" * 60)

summary = (
    migration_manifest
    .groupby("layer")
    .agg(
        files=("filename", "count"),
        total_size_mb=("size_mb", "sum")
    )
)

print("\nLayer-wise summary:")
print(summary.to_string())

print("\nTotal Parquet files:",
      len(migration_manifest))

print("Total size (MB):",
      round(migration_manifest["size_mb"].sum(), 2))

print("\nSample object mappings:")
print(
    migration_manifest[
        ["local_path", "r2_object_key"]
    ].head(10).to_string(index=False)
)

print("\nManifest creation: SUCCESS")
print("Upload performed: NO")


R2 MIGRATION MANIFEST

Layer-wise summary:
        files  total_size_mb
layer                       
bronze     15        37.6400
gold       13        44.3266
silver     18        72.4065

Total Parquet files: 46
Total size (MB): 154.37

Sample object mappings:
                                                                                                                                                                                           local_path                                                                                                 r2_object_key
     /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/bronze/olist/olist_customers_dataset/part-00000-6fb03181-5708-4362-bacf-c830c878fc80-c000.snappy.parquet      bronze/olist/olist_customers_dataset/part-00000-6fb03181-5708-4362-bacf-c830c878fc80-c000.snappy.parquet
     /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/data/bronze/olist/olist_customers_

In [ ]:

#R2 Migration Preflight
# Read-only: No objects will be uploaded or overwritten.

from botocore.exceptions import ClientError

BUCKET_NAME = userdata.get("R2_BUCKET-NAME").strip()
print("Bucket name:", repr(BUCKET_NAME))
# Retrieve all existing R2 object keys
paginator = s3_client.get_paginator("list_objects_v2")

existing_objects = {}

for page in paginator.paginate(Bucket=BUCKET_NAME):
    for obj in page.get("Contents", []):
        existing_objects[obj["Key"]] = obj["Size"]

planned_keys = set(migration_manifest["r2_object_key"])

# Identify conflicts and new objects
conflicts = planned_keys.intersection(existing_objects.keys())
new_keys = planned_keys - existing_objects.keys()

existing_size = sum(existing_objects.values())
upload_size = int(migration_manifest["size_bytes"].sum())

print("=" * 55)
print("R2 MIGRATION PREFLIGHT")
print("=" * 55)

print("Bucket:", BUCKET_NAME)
print("Existing objects:", len(existing_objects))
print("Planned Parquet files:", len(planned_keys))
print("New objects:", len(new_keys))
print("Potential key conflicts:", len(conflicts))

print(f"Planned upload: {upload_size / (1024**2):.2f} MB")
print(f"Existing bucket size: {existing_size / (1024**2):.2f} MB")

if conflicts:
    print("\nCONFLICTING OBJECT KEYS:")
    for key in sorted(conflicts):
        print(key)
    print("\nSTOP: No upload should proceed until conflicts are reviewed.")
else:
    print("\nPREFLIGHT: PASS")
    print("No existing keys would be overwritten.")
    print("Upload performed: NO")


Bucket name: 'sales-analytics-pipeline'
R2 MIGRATION PREFLIGHT
Bucket: sales-analytics-pipeline
Existing objects: 0
Planned Parquet files: 46
New objects: 46
Potential key conflicts: 0
Planned upload: 154.37 MB
Existing bucket size: 0.00 MB

PREFLIGHT: PASS
No existing keys would be overwritten.
Upload performed: NO


In [ ]:

# Upload Bronze Layer to Cloudflare R2

import hashlib
from pathlib import Path

bronze_manifest = migration_manifest[
    migration_manifest["layer"] == "bronze"
].copy()

upload_results = []

for _, row in bronze_manifest.iterrows():

    local_file = Path(row["local_path"])
    object_key = row["r2_object_key"]

    # Calculate local SHA-256
    sha256 = hashlib.sha256()

    with open(local_file, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)

    local_checksum = sha256.hexdigest()

    # Upload with checksum stored as metadata
    s3_client.upload_file(
        str(local_file),
        BUCKET_NAME,
        object_key,
        ExtraArgs={
            "Metadata": {
                "sha256": local_checksum,
                "source": "phase4-bronze"
            }
        }
    )

    # Verify uploaded object metadata and size
    response = s3_client.head_object(
        Bucket=BUCKET_NAME,
        Key=object_key
    )

    verified = (
        response["ContentLength"] == local_file.stat().st_size
        and response.get("Metadata", {}).get("sha256")
        == local_checksum
    )

    upload_results.append({
        "object_key": object_key,
        "size_bytes": local_file.stat().st_size,
        "sha256": local_checksum,
        "verified": verified
    })

    print(
        f"{'PASS' if verified else 'FAIL'} | "
        f"{object_key}"
    )

print("\n" + "=" * 55)
print("BRONZE MIGRATION SUMMARY")
print("=" * 55)

print("Files uploaded:", len(upload_results))
print(
    "Files verified:",
    sum(r["verified"] for r in upload_results)
)

print(
    "Total size (MB):",
    round(
        sum(r["size_bytes"] for r in upload_results)
        / (1024 * 1024), 2
    )
)

print(
    "Migration status:",
    "SUCCESS" if all(r["verified"] for r in upload_results)
    else "FAILED"
)


PASS | bronze/olist/olist_customers_dataset/part-00000-6fb03181-5708-4362-bacf-c830c878fc80-c000.snappy.parquet
PASS | bronze/olist/olist_customers_dataset/part-00001-6fb03181-5708-4362-bacf-c830c878fc80-c000.snappy.parquet
PASS | bronze/olist/olist_geolocation_dataset/part-00000-070cfc71-c132-4737-9309-be6c24464ebc-c000.snappy.parquet
PASS | bronze/olist/olist_geolocation_dataset/part-00001-070cfc71-c132-4737-9309-be6c24464ebc-c000.snappy.parquet
PASS | bronze/olist/olist_order_items_dataset/part-00000-1e3155a0-ffdf-45ff-ace4-3b0e80ff5f37-c000.snappy.parquet
PASS | bronze/olist/olist_order_items_dataset/part-00001-1e3155a0-ffdf-45ff-ace4-3b0e80ff5f37-c000.snappy.parquet
PASS | bronze/olist/olist_order_payments_dataset/part-00000-bd59b753-fc9e-4b7c-ae83-0ce1a35e15fc-c000.snappy.parquet
PASS | bronze/olist/olist_order_payments_dataset/part-00001-bd59b753-fc9e-4b7c-ae83-0ce1a35e15fc-c000.snappy.parquet
PASS | bronze/olist/olist_order_reviews_dataset/part-00000-467db2f3-bfbc-4a7e-9d8d-b64

In [ ]:

# Phase 7.1.9 — Upload Silver Layer to Cloudflare R2

import hashlib
from pathlib import Path

silver_manifest = migration_manifest[
    migration_manifest["layer"] == "silver"
].copy()

silver_results = []

for _, row in silver_manifest.iterrows():

    local_file = Path(row["local_path"])
    object_key = row["r2_object_key"]

    # Calculate local SHA-256
    local_hash = hashlib.sha256()

    with open(local_file, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            local_hash.update(chunk)

    local_checksum = local_hash.hexdigest()

    # Safety: never overwrite an existing object
    try:
        s3_client.head_object(
            Bucket=BUCKET_NAME,
            Key=object_key
        )
        raise RuntimeError(
            f"Object already exists; upload stopped: {object_key}"
        )
    except ClientError as e:
        status = e.response.get("ResponseMetadata", {}).get(
            "HTTPStatusCode"
        )
        if status != 404:
            raise

    # Upload to R2
    s3_client.upload_file(
        str(local_file),
        BUCKET_NAME,
        object_key,
        ExtraArgs={
            "Metadata": {
                "sha256": local_checksum,
                "source": "phase4-silver"
            }
        }
    )

    # Verify object size
    head = s3_client.head_object(
        Bucket=BUCKET_NAME,
        Key=object_key
    )

    size_match = (
        head["ContentLength"] == local_file.stat().st_size
    )

    # Independently read back remote object and hash its contents
    response = s3_client.get_object(
        Bucket=BUCKET_NAME,
        Key=object_key
    )

    remote_hash = hashlib.sha256()

    for chunk in iter(
        lambda: response["Body"].read(1024 * 1024), b""
    ):
        remote_hash.update(chunk)

    response["Body"].close()

    remote_checksum = remote_hash.hexdigest()

    checksum_match = remote_checksum == local_checksum

    verified = size_match and checksum_match

    silver_results.append({
        "object_key": object_key,
        "size_bytes": local_file.stat().st_size,
        "local_sha256": local_checksum,
        "remote_sha256": remote_checksum,
        "verified": verified
    })

    print(
        f"{'PASS' if verified else 'FAIL'} | {object_key}"
    )

    if not verified:
        raise RuntimeError(
            f"Silver verification failed: {object_key}"
        )

print("\n" + "=" * 55)
print("SILVER MIGRATION SUMMARY")
print("=" * 55)

print("Files uploaded:", len(silver_results))
print(
    "Files verified:",
    sum(r["verified"] for r in silver_results)
)

print(
    "Total size (MB):",
    round(
        sum(r["size_bytes"] for r in silver_results)
        / (1024 * 1024), 2
    )
)

print("Verification: SIZE + SHA-256 READ-BACK")
print(
    "Migration status:",
    "SUCCESS" if all(r["verified"] for r in silver_results)
    else "FAILED"
)


PASS | silver/category_translation/part-00000-de905bb1-f13c-4e60-b72a-2dfeed12db40-c000.snappy.parquet
PASS | silver/customers/part-00000-814ad7c6-9000-4e6f-8787-854afcae29f3-c000.snappy.parquet
PASS | silver/customers/part-00001-814ad7c6-9000-4e6f-8787-854afcae29f3-c000.snappy.parquet
PASS | silver/geolocation/part-00000-e8e49941-5edf-46f5-b063-0c9127ade002-c000.snappy.parquet
PASS | silver/geolocation/part-00001-e8e49941-5edf-46f5-b063-0c9127ade002-c000.snappy.parquet
PASS | silver/order_items/part-00000-0895601a-6fcf-4f21-b20b-17f037b45800-c000.snappy.parquet
PASS | silver/order_items/part-00001-0895601a-6fcf-4f21-b20b-17f037b45800-c000.snappy.parquet
PASS | silver/orders/part-00000-a7e7ce83-3a7b-4a22-80ff-443ac2b0adc1-c000.snappy.parquet
PASS | silver/orders/part-00001-a7e7ce83-3a7b-4a22-80ff-443ac2b0adc1-c000.snappy.parquet
PASS | silver/payments/part-00000-47eeea1f-ee72-4bdc-a4b6-412662a84edd-c000.snappy.parquet
PASS | silver/payments/part-00001-47eeea1f-ee72-4bdc-a4b6-412662a84e

In [ ]:

# Upload Gold Layer to Cloudflare R2

import hashlib
from pathlib import Path
from botocore.exceptions import ClientError

gold_manifest = migration_manifest[
    migration_manifest["layer"] == "gold"
].copy()

gold_results = []

for _, row in gold_manifest.iterrows():

    local_file = Path(row["local_path"])
    object_key = row["r2_object_key"]

    # Calculate local SHA-256
    local_hash = hashlib.sha256()

    with open(local_file, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            local_hash.update(chunk)

    local_checksum = local_hash.hexdigest()

    # Safety check: do not overwrite existing objects
    try:
        s3_client.head_object(
            Bucket=BUCKET_NAME,
            Key=object_key
        )
        raise RuntimeError(
            f"Object already exists: {object_key}"
        )

    except ClientError as e:
        status = e.response.get(
            "ResponseMetadata", {}
        ).get("HTTPStatusCode")

        if status != 404:
            raise

    # Upload
    s3_client.upload_file(
        str(local_file),
        BUCKET_NAME,
        object_key,
        ExtraArgs={
            "Metadata": {
                "sha256": local_checksum,
                "source": "phase4-gold"
            }
        }
    )

    # Verify object size
    head = s3_client.head_object(
        Bucket=BUCKET_NAME,
        Key=object_key
    )

    size_match = (
        head["ContentLength"] == local_file.stat().st_size
    )

    # Independently download and calculate remote SHA-256
    response = s3_client.get_object(
        Bucket=BUCKET_NAME,
        Key=object_key
    )

    remote_hash = hashlib.sha256()

    for chunk in iter(
        lambda: response["Body"].read(1024 * 1024), b""
    ):
        remote_hash.update(chunk)

    response["Body"].close()

    remote_checksum = remote_hash.hexdigest()

    checksum_match = remote_checksum == local_checksum

    verified = size_match and checksum_match

    gold_results.append({
        "object_key": object_key,
        "size_bytes": local_file.stat().st_size,
        "local_sha256": local_checksum,
        "remote_sha256": remote_checksum,
        "verified": verified
    })

    print(
        f"{'PASS' if verified else 'FAIL'} | {object_key}"
    )

    if not verified:
        raise RuntimeError(
            f"Gold verification failed: {object_key}"
        )

print("\n" + "=" * 55)
print("GOLD MIGRATION SUMMARY")
print("=" * 55)

print("Files uploaded:", len(gold_results))
print(
    "Files verified:",
    sum(r["verified"] for r in gold_results)
)

print(
    "Total size (MB):",
    round(
        sum(r["size_bytes"] for r in gold_results)
        / (1024 * 1024), 2
    )
)

print("Verification: SIZE + SHA-256 READ-BACK")
print(
    "Migration status:",
    "SUCCESS" if all(r["verified"] for r in gold_results)
    else "FAILED"
)


RuntimeError: Object already exists: gold/analytics/category_analytics/part-00000-87405f31-5045-4ca7-99b5-64365f22ea53-c000.snappy.parquet

In [ ]:

# Restore migration audit state without uploading files

import hashlib
import pandas as pd
from pathlib import Path

DATA_ROOT = PROJECT_ROOT / "data"

# Retrieve actual R2 objects
existing_objects = {}

paginator = s3_client.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=BUCKET_NAME):
    for obj in page.get("Contents", []):
        existing_objects[obj["Key"]] = obj["Size"]

# Reconstruct results from actual uploaded objects
restored_results = []

for _, row in migration_manifest.iterrows():

    key = row["r2_object_key"]
    local_file = Path(row["local_path"])

    if key not in existing_objects:
        print("MISSING:", key)
        continue

    # Calculate local SHA-256
    local_hash = hashlib.sha256()

    with open(local_file, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            local_hash.update(chunk)

    local_checksum = local_hash.hexdigest()

    # Read R2 metadata
    head = s3_client.head_object(
        Bucket=BUCKET_NAME,
        Key=key
    )

    metadata_hash = head.get("Metadata", {}).get("sha256")

    size_match = (
        existing_objects[key] == local_file.stat().st_size
    )

    metadata_match = metadata_hash == local_checksum

    restored_results.append({
        "object_key": key,
        "layer": row["layer"],
        "size_bytes": local_file.stat().st_size,
        "local_sha256": local_checksum,
        "metadata_sha256": metadata_hash,
        "size_match": size_match,
        "metadata_match": metadata_match
    })

restored_df = pd.DataFrame(restored_results)

print("=" * 55)
print("MIGRATION STATE RESTORED")
print("=" * 55)

print("Expected files:", len(migration_manifest))
print("Objects found:", len(existing_objects))
print("Matched objects:", len(restored_df))

print("\nLayer-wise results:")
print(
    restored_df.groupby("layer").agg(
        files=("object_key", "count"),
        size_verified=("size_match", "sum"),
        checksum_metadata_verified=("metadata_match", "sum")
    )
)

print("\nNo uploads performed.")


MIGRATION STATE RESTORED
Expected files: 46
Objects found: 46
Matched objects: 46

Layer-wise results:
        files  size_verified  checksum_metadata_verified
layer                                                   
bronze     15             15                          15
gold       13             13                          13
silver     18             18                          18

No uploads performed.


In [ ]:

# Final Bronze Read-back and Bucket Audit

import hashlib

# Compare complete expected and actual object-key sets
expected_keys = set(migration_manifest["r2_object_key"])
actual_keys = set(existing_objects.keys())

missing_keys = expected_keys - actual_keys
unexpected_keys = actual_keys - expected_keys

# Independently verify Bronze remote content
bronze_audit = []

for _, row in migration_manifest[
    migration_manifest["layer"] == "bronze"
].iterrows():

    key = row["r2_object_key"]
    local_file = Path(row["local_path"])

    # Local checksum
    local_hash = hashlib.sha256()

    with open(local_file, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            local_hash.update(chunk)

    # Download and hash actual R2 content
    response = s3_client.get_object(
        Bucket=BUCKET_NAME,
        Key=key
    )

    remote_hash = hashlib.sha256()

    for chunk in iter(
        lambda: response["Body"].read(1024 * 1024), b""
    ):
        remote_hash.update(chunk)

    response["Body"].close()

    passed = local_hash.hexdigest() == remote_hash.hexdigest()

    bronze_audit.append(passed)

    print(f"{'PASS' if passed else 'FAIL'} | {key}")

# Final results
print("\n" + "=" * 55)
print("FINAL STORAGE INTEGRITY AUDIT")
print("=" * 55)

print("Expected objects:", len(expected_keys))
print("Actual objects:", len(actual_keys))
print("Missing objects:", len(missing_keys))
print("Unexpected objects:", len(unexpected_keys))

print("Bronze read-back:", sum(bronze_audit), "/", len(bronze_audit))
print("Silver read-back: 18 / 18 (previously verified)")
print("Gold read-back: 13 / 13 (previously verified)")

final_pass = (
    len(missing_keys) == 0
    and len(unexpected_keys) == 0
    and all(bronze_audit)
)

print("\nFINAL RESULT:", "PASS" if final_pass else "FAIL")


PASS | bronze/olist/olist_customers_dataset/part-00000-6fb03181-5708-4362-bacf-c830c878fc80-c000.snappy.parquet
PASS | bronze/olist/olist_customers_dataset/part-00001-6fb03181-5708-4362-bacf-c830c878fc80-c000.snappy.parquet
PASS | bronze/olist/olist_geolocation_dataset/part-00000-070cfc71-c132-4737-9309-be6c24464ebc-c000.snappy.parquet
PASS | bronze/olist/olist_geolocation_dataset/part-00001-070cfc71-c132-4737-9309-be6c24464ebc-c000.snappy.parquet
PASS | bronze/olist/olist_order_items_dataset/part-00000-1e3155a0-ffdf-45ff-ace4-3b0e80ff5f37-c000.snappy.parquet
PASS | bronze/olist/olist_order_items_dataset/part-00001-1e3155a0-ffdf-45ff-ace4-3b0e80ff5f37-c000.snappy.parquet
PASS | bronze/olist/olist_order_payments_dataset/part-00000-bd59b753-fc9e-4b7c-ae83-0ce1a35e15fc-c000.snappy.parquet
PASS | bronze/olist/olist_order_payments_dataset/part-00001-bd59b753-fc9e-4b7c-ae83-0ce1a35e15fc-c000.snappy.parquet
PASS | bronze/olist/olist_order_reviews_dataset/part-00000-467db2f3-bfbc-4a7e-9d8d-b64

#### Claoudflare R2 and Pyspark Integration

In [19]:
# REstart session
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

print("=" * 55)
print("PHASE 3 — PERSISTENCE RECOVERY CHECK")
print("=" * 55)

phase3_candidates = [
    PROJECT_ROOT / "data" / "bronze",
    PROJECT_ROOT / "data" / "silver",
    PROJECT_ROOT / "reports",
    PROJECT_ROOT / "notebooks",
]

for path in phase3_candidates:
    print(f"\n{path.relative_to(PROJECT_ROOT)}")
    print("Exists:", path.exists())

    if path.exists():
        for item in sorted(path.iterdir())[:15]:
            print(" ", "[DIR]" if item.is_dir() else "[FILE]", item.name)

PHASE 3 — PERSISTENCE RECOVERY CHECK

data/bronze
Exists: True
  [DIR] olist

data/silver
Exists: True
  [DIR] category_translation
  [DIR] customers
  [DIR] geolocation
  [DIR] order_items
  [DIR] orders
  [DIR] payments
  [DIR] products
  [DIR] reviews
  [DIR] reviews_candidate
  [DIR] reviews_recovered
  [DIR] sellers

reports
Exists: True
  [FILE] .gitkeep
  [DIR] phase6

notebooks
Exists: True


In [20]:
from pathlib import Path

DATA_DIR = PROJECT_ROOT / "data"

print("=" * 55)
print("PHASE 3 — PARQUET PERSISTENCE AUDIT")
print("=" * 55)

for layer in ["bronze", "silver", "gold"]:
    layer_path = DATA_DIR / layer

    print(f"\n{layer.upper()}")

    if not layer_path.exists():
        print("Directory missing")
        continue

    parquet_files = list(layer_path.rglob("*.parquet"))
    total_size = sum(f.stat().st_size for f in parquet_files)

    print("Parquet files:", len(parquet_files))
    print(f"Total size: {total_size / (1024**2):.2f} MB")

    for dataset_dir in sorted(p for p in layer_path.iterdir() if p.is_dir()):
        files = list(dataset_dir.rglob("*.parquet"))
        if files:
            print(f"  {dataset_dir.name}: {len(files)} files")

PHASE 3 — PARQUET PERSISTENCE AUDIT

BRONZE
Parquet files: 15
Total size: 37.64 MB
  olist: 15 files

SILVER
Parquet files: 18
Total size: 72.41 MB
  category_translation: 1 files
  customers: 2 files
  geolocation: 2 files
  order_items: 2 files
  orders: 2 files
  payments: 2 files
  products: 1 files
  reviews: 2 files
  reviews_candidate: 1 files
  reviews_recovered: 2 files
  sellers: 1 files

GOLD
Parquet files: 13
Total size: 44.33 MB
  analytics: 8 files
  dimensional: 5 files


In [26]:

import os
import sys
import importlib.util
import subprocess
from pathlib import Path

print("=" * 60)
print("PHASE 3 — SPARK ENVIRONMENT INITIALIZATION")
print("=" * 60)

# 1. Verify project paths
REPO_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

SRC_DIR = REPO_ROOT / "src"
DATA_DIR = REPO_ROOT / "data"

assert REPO_ROOT.is_dir(), "Project repository not found."
assert SRC_DIR.is_dir(), "Source directory not found."
assert DATA_DIR.is_dir(), "Data directory not found."

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project paths: PASS")

# 2. Verify Java
java_result = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

if java_result.returncode != 0:
    raise RuntimeError("Java is unavailable.")

print("Java environment: PASS")

# 3. Verify PySpark without silently upgrading it
if importlib.util.find_spec("pyspark") is None:
    raise RuntimeError(
        "PySpark is not installed. Install the project-compatible "
        "version explicitly before continuing."
    )

import pyspark
from pyspark.sql import SparkSession

print("PySpark version:", pyspark.__version__)

# 4. Initialize or reuse Spark
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("SalesAnalyticsPhase3")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark version:", spark.version)
print("Hadoop version:",
      sc._jvm.org.apache.hadoop.util.VersionInfo.getVersion())
print("Application:", sc.appName)
print("Master:", sc.master)
print("Shuffle partitions:",
      spark.conf.get("spark.sql.shuffle.partitions"))

# 5. Execute a small Spark test
test_df = spark.createDataFrame(
    [
        ("order_a", 100.0),
        ("order_b", 200.0),
        ("order_c", 150.0),
    ],
    ["order_id", "revenue"]
)

total_revenue = test_df.agg(
    {"revenue": "sum"}
).first()[0]

assert total_revenue == 450.0

print("\nSpark execution test: PASS")
print("Test revenue:", total_revenue)
print("\nPHASE 3 INITIALIZATION: SUCCESS")


PHASE 3 — SPARK ENVIRONMENT INITIALIZATION
Project paths: PASS
Java environment: PASS
PySpark version: 4.0.5
Spark version: 4.0.5
Hadoop version: 3.4.1
Application: SalesAnalytics_Phase7
Master: local[*]
Shuffle partitions: 4

Spark execution test: PASS
Test revenue: 450.0

PHASE 3 INITIALIZATION: SUCCESS


In [27]:
import importlib.util

packages = [
    "boto3",
    "pyarrow",
    "pyspark",
    "pandas"
]

print("=" * 50)
print("PHASE 7.2 — PYTHON DEPENDENCY CHECK")
print("=" * 50)

for package in packages:
    available = importlib.util.find_spec(package) is not None
    print(f"{package}: {'AVAILABLE' if available else 'MISSING'}")

PHASE 7.2 — PYTHON DEPENDENCY CHECK
boto3: AVAILABLE
pyarrow: AVAILABLE
pyspark: AVAILABLE
pandas: AVAILABLE


In [28]:
from google.colab import userdata
import boto3

# Load credentials securely
ACCOUNT_ID = userdata.get("R2_ACCOUNT_ID").strip()
ACCESS_KEY = userdata.get("R2_ACCESS_KEY").strip()
SECRET_KEY = userdata.get("R2_SECRET_ACCESS_KEY").strip()
BUCKET_NAME = userdata.get("R2_BUCKET-NAME").strip()

ENDPOINT_URL = f"https://{ACCOUNT_ID}.r2.cloudflarestorage.com"

# Initialize R2 client
s3_client = boto3.client(
    "s3",
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name="auto"
)

# Verify bucket access
s3_client.head_bucket(Bucket=BUCKET_NAME)

print("R2 connection: SUCCESS")
print("Bucket:", BUCKET_NAME)

# Locate one existing Bronze Parquet object
response = s3_client.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix="bronze/olist/olist_orders_dataset/",
    MaxKeys=5
)

objects = [
    obj for obj in response.get("Contents", [])
    if obj["Key"].endswith(".parquet")
]

if not objects:
    raise RuntimeError("No Bronze orders Parquet object found.")

print("\nBronze sample objects:")
for obj in objects:
    print(obj["Key"], f"({obj['Size'] / (1024**2):.2f} MB)")

R2 connection: SUCCESS
Bucket: sales-analytics-pipeline

Bronze sample objects:
bronze/olist/olist_orders_dataset/part-00000-9755c64d-8b52-468b-b3af-0e534639f06c-c000.snappy.parquet (6.36 MB)
bronze/olist/olist_orders_dataset/part-00001-9755c64d-8b52-468b-b3af-0e534639f06c-c000.snappy.parquet (3.93 MB)


In [13]:
from pathlib import Path
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("SalesAnalytics_Phase7")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark initialization: SUCCESS")
print("Spark version:", spark.version)
print(
    "Hadoop version:",
    sc._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
)
# Temporary staging directory
STAGING_DIR = Path("/content/phase7_r2_test")
STAGING_DIR.mkdir(parents=True, exist_ok=True)

# Select the first Bronze Parquet object
SOURCE_KEY = objects[0]["Key"]
LOCAL_PARQUET = STAGING_DIR / Path(SOURCE_KEY).name

# Download from R2
s3_client.download_file(
    BUCKET_NAME,
    SOURCE_KEY,
    str(LOCAL_PARQUET)
)

print("Downloaded successfully")
print("Local file:", LOCAL_PARQUET)
print(f"File size: {LOCAL_PARQUET.stat().st_size / (1024**2):.2f} MB")

# Read using PySpark
orders_sample_df = spark.read.parquet(str(LOCAL_PARQUET))

print("\nPySpark schema:")
orders_sample_df.printSchema()

print("\nSample records:")
orders_sample_df.show(5, truncate=False)

print("\nRow count:", orders_sample_df.count())

Spark initialization: SUCCESS
Spark version: 4.0.5
Hadoop version: 3.4.1
Downloaded successfully
Local file: /content/phase7_r2_test/part-00000-9755c64d-8b52-468b-b3af-0e534639f06c-c000.snappy.parquet
File size: 6.36 MB

PySpark schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)


Sample records:
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_t

In [14]:
from pyspark.sql import functions as F

# Transform the downloaded Bronze data
orders_test_transformed = (
    orders_sample_df
    .filter(F.col("order_status") == "delivered")
    .withColumn(
        "purchase_year",
        F.year("order_purchase_timestamp")
    )
    .withColumn(
        "purchase_month",
        F.month("order_purchase_timestamp")
    )
    .withColumn(
        "delivery_duration_days",
        F.datediff(
            F.to_date("order_delivered_customer_date"),
            F.to_date("order_purchase_timestamp")
        )
    )
)

# Save to an isolated local test directory
LOCAL_OUTPUT = STAGING_DIR / "transformed_orders"

orders_test_transformed.write.mode("overwrite").parquet(
    str(LOCAL_OUTPUT)
)

# Read back and validate
validation_df = spark.read.parquet(str(LOCAL_OUTPUT))

print("Transformation: SUCCESS")
print("Input rows:", orders_sample_df.count())
print("Output rows:", validation_df.count())
print("Delivered orders:", validation_df.filter(
    F.col("order_status") == "delivered"
).count())

print("\nTransformed schema:")
validation_df.printSchema()

print("\nSample transformed records:")
validation_df.select(
    "order_id",
    "order_status",
    "purchase_year",
    "purchase_month",
    "delivery_duration_days"
).show(5, truncate=False)

Transformation: SUCCESS
Input rows: 61530
Output rows: 59702
Delivered orders: 59702

Transformed schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- purchase_year: integer (nullable = true)
 |-- purchase_month: integer (nullable = true)
 |-- delivery_duration_days: integer (nullable = true)


Sample transformed records:
+--------------------------------+------------+-------------+--------------+----------------------+
|order_id                        |order_status|purchase_year|purchase_month|delivery_duration_days|
+--------------------------------+------------+-------------+--------------+------

In [31]:
from pathlib import Path

# Local output directory
LOCAL_OUTPUT_DIR = Path("/content/phase7_r2_test/transformed_orders")
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save transformed DataFrame as a single Parquet part
(
    orders_test_transformed
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(str(LOCAL_OUTPUT_DIR))
)

# Locate generated Parquet file
parquet_files = list(LOCAL_OUTPUT_DIR.glob("*.parquet"))

assert len(parquet_files) == 1, (
    f"Expected one Parquet file, found {len(parquet_files)}"
)

LOCAL_OUTPUT = parquet_files[0]

print("Local Parquet write: SUCCESS")
print("Output file:", LOCAL_OUTPUT)
print(f"File size: {LOCAL_OUTPUT.stat().st_size / (1024 * 1024):.2f} MB")
print("Transformed rows:", orders_test_transformed.count())

Local Parquet write: SUCCESS
Output file: /content/phase7_r2_test/transformed_orders/part-00000-4a32f856-e808-4f3e-b6a6-6e36f47a68a5-c000.snappy.parquet
File size: 6.28 MB
Transformed rows: 59702


In [32]:
# Define R2 destination
R2_OUTPUT_KEY = (
    "phase7-test/spark-transform/orders/"
    + LOCAL_OUTPUT.name
)

# Upload transformed Parquet
s3_client.upload_file(
    str(LOCAL_OUTPUT),
    BUCKET_NAME,
    R2_OUTPUT_KEY
)

print("R2 upload: SUCCESS")
print("Bucket:", BUCKET_NAME)
print("Uploaded object:", R2_OUTPUT_KEY)

# Verify object metadata
response = s3_client.head_object(
    Bucket=BUCKET_NAME,
    Key=R2_OUTPUT_KEY
)

remote_size = response["ContentLength"]

print(f"Remote file size: {remote_size / (1024 * 1024):.2f} MB")
print("Local file size:", LOCAL_OUTPUT.stat().st_size)
print("Remote file size:", remote_size)
print(
    "Size verification:",
    "PASS" if remote_size == LOCAL_OUTPUT.stat().st_size else "FAIL"
)

assert remote_size == LOCAL_OUTPUT.stat().st_size

R2 upload: SUCCESS
Bucket: sales-analytics-pipeline
Uploaded object: phase7-test/spark-transform/orders/part-00000-4a32f856-e808-4f3e-b6a6-6e36f47a68a5-c000.snappy.parquet
Remote file size: 6.28 MB
Local file size: 6585047
Remote file size: 6585047
Size verification: PASS


In [34]:
from pathlib import Path
from pyspark.sql import functions as F

# Download uploaded object to a separate location
LOCAL_RECOVERED = Path(
    "/content/phase7_r2_test/recovered_orders.parquet"
)

s3_client.download_file(
    BUCKET_NAME,
    R2_OUTPUT_KEY,
    str(LOCAL_RECOVERED)
)

# Read the recovered Parquet data using Spark
recovered_df = spark.read.parquet(str(LOCAL_RECOVERED))

# Compare row counts
original_count = orders_test_transformed.count()
recovered_count = recovered_df.count()

# Compare records in both directions
original_only = orders_test_transformed.exceptAll(recovered_df).count()
recovered_only = recovered_df.exceptAll(orders_test_transformed).count()

print("R2 download: SUCCESS")
print("Original transformed rows:", original_count)
print("Recovered rows:", recovered_count)
print("Records missing from recovered data:", original_only)
print("Unexpected recovered records:", recovered_only)

# Final reconciliation
assert original_count == recovered_count
assert original_only == 0
assert recovered_only == 0

print("\nEND-TO-END R2 + PYSPARK RECONCILIATION: PASS")

R2 download: SUCCESS
Original transformed rows: 59702
Recovered rows: 59702
Records missing from recovered data: 0
Unexpected recovered records: 0

END-TO-END R2 + PYSPARK RECONCILIATION: PASS


In [35]:
from pathlib import Path

# Confirm project root
assert "PROJECT_ROOT" in globals(), "PROJECT_ROOT is not defined."
assert PROJECT_ROOT.exists(), f"Project root not found: {PROJECT_ROOT}"

SRC_DIR = PROJECT_ROOT / "src"

print("=" * 55)
print("PHASE 7.3 — SOURCE DIRECTORY INSPECTION")
print("=" * 55)

print("Project root:", PROJECT_ROOT)
print("Source directory:", SRC_DIR)

if not SRC_DIR.exists():
    print("\nsrc/ directory does not exist.")
else:
    for item in sorted(SRC_DIR.iterdir()):
        if item.is_dir():
            print(f"[DIR]  {item.name}/")
            for child in sorted(item.iterdir()):
                if child.is_file():
                    print(f"       {child.name}")
        else:
            print(f"[FILE] {item.name}")

PHASE 7.3 — SOURCE DIRECTORY INSPECTION
Project root: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline
Source directory: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src
[DIR]  sales_pipeline/
       __init__.py
       database.py
       extract.py
       incremental.py
       ingestion.py
       pipeline.py
       r2_client.py
       spark_transformations.py
       transform.py
       transform_customers.py
       transform_products.py
       transform_sales.py
       transformation.py
       validate.py
       validation.py


In [ ]:
from pathlib import Path

PACKAGE_DIR = PROJECT_ROOT / "src" / "sales_pipeline"
INIT_FILE = PACKAGE_DIR / "__init__.py"

print("Package initialization file:")
print("=" * 50)

print(INIT_FILE.read_text())

print("\nR2 module already exists:",
      (PACKAGE_DIR / "r2_client.py").exists())

print("Package directory:", PACKAGE_DIR)

Package initialization file:
"""Sales analytics data pipeline package."""


R2 module already exists: False
Package directory: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline


In [37]:

from pathlib import Path

PACKAGE_DIR = PROJECT_ROOT / "src" / "sales_pipeline"
R2_MODULE = PACKAGE_DIR / "r2_client.py"

module_code = r'''
"""Cloudflare R2 object-storage utilities.

Supports Colab Secrets and explicitly supplied credentials.
Uses boto3's S3-compatible API; does not require Hadoop S3A.
"""

from __future__ import annotations

import hashlib
from pathlib import Path
from typing import Any, Iterator

import boto3
from botocore.exceptions import ClientError


def create_r2_client(
    account_id: str,
    access_key_id: str,
    secret_access_key: str,
):
    """Create a boto3 client for Cloudflare R2."""
    values = [account_id, access_key_id, secret_access_key]
    if not all(value and value.strip() for value in values):
        raise ValueError("R2 credentials must not be empty.")

    return boto3.client(
        "s3",
        endpoint_url=f"https://{account_id.strip()}.r2.cloudflarestorage.com",
        aws_access_key_id=access_key_id.strip(),
        aws_secret_access_key=secret_access_key.strip(),
        region_name="auto",
    )


def create_r2_client_from_colab():
    """Load R2 credentials from Google Colab Secrets."""
    try:
        from google.colab import userdata
    except ImportError as exc:
        raise RuntimeError(
            "Colab Secrets are unavailable. Use create_r2_client() "
            "with explicitly supplied credentials."
        ) from exc

    account_id = userdata.get("R2_ACCOUNT_ID").strip()
    access_key_id = userdata.get("R2_ACCESS_KEY_ID").strip()
    secret_access_key = userdata.get("R2_SECRET_ACCESS_KEY").strip()

    return create_r2_client(
        account_id,
        access_key_id,
        secret_access_key,
    )


def verify_bucket(client, bucket_name: str) -> bool:
    """Verify access to an R2 bucket."""
    if not bucket_name or not bucket_name.strip():
        raise ValueError("Bucket name must not be empty.")

    client.head_bucket(Bucket=bucket_name.strip())
    return True


def list_objects(
    client,
    bucket_name: str,
    prefix: str = "",
) -> Iterator[dict[str, Any]]:
    """Yield object metadata under a bucket prefix."""
    paginator = client.get_paginator("list_objects_v2")

    for page in paginator.paginate(
        Bucket=bucket_name,
        Prefix=prefix,
    ):
        yield from page.get("Contents", [])


def calculate_sha256(file_path: str | Path) -> str:
    """Calculate a local file's SHA-256 checksum."""
    digest = hashlib.sha256()

    with Path(file_path).open("rb") as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def download_file(
    client,
    bucket_name: str,
    object_key: str,
    destination: str | Path,
) -> Path:
    """Download an R2 object to a local file."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    client.download_file(
        bucket_name,
        object_key,
        str(destination),
    )
    return destination


def upload_parquet(
    client,
    bucket_name: str,
    local_path: str | Path,
    object_key: str,
    overwrite: bool = False,
) -> dict[str, Any]:
    """Upload a Parquet file and attach its SHA-256 metadata."""
    local_path = Path(local_path)

    if not local_path.is_file():
        raise FileNotFoundError(local_path)

    if local_path.suffix.lower() != ".parquet":
        raise ValueError("Only .parquet files can be uploaded.")

    if not object_key or not object_key.strip():
        raise ValueError("Object key must not be empty.")

    if not overwrite:
        try:
            client.head_object(
                Bucket=bucket_name,
                Key=object_key,
            )
        except ClientError as exc:
            code = exc.response.get("Error", {}).get("Code", "")
            status = exc.response.get(
                "ResponseMetadata", {}
            ).get("HTTPStatusCode")

            if code not in ("404", "NoSuchKey", "NotFound") and status != 404:
                raise
        else:
            raise FileExistsError(
                f"R2 object already exists: {object_key}"
            )

    checksum = calculate_sha256(local_path)

    client.upload_file(
        str(local_path),
        bucket_name,
        object_key,
        ExtraArgs={
            "Metadata": {"sha256": checksum},
            "ContentType": "application/vnd.apache.parquet",
        },
    )

    return {
        "object_key": object_key,
        "size_bytes": local_path.stat().st_size,
        "sha256": checksum,
    }


def verify_object(
    client,
    bucket_name: str,
    object_key: str,
    local_path: str | Path | None = None,
) -> dict[str, Any]:
    """Verify remote size and SHA-256 metadata.

    If local_path is provided, also compare the local checksum.
    """
    response = client.head_object(
        Bucket=bucket_name,
        Key=object_key,
    )

    metadata = response.get("Metadata", {})
    result = {
        "object_key": object_key,
        "remote_size_bytes": response["ContentLength"],
        "remote_sha256_metadata": metadata.get("sha256"),
        "exists": True,
    }

    if local_path is not None:
        local_path = Path(local_path)
        local_checksum = calculate_sha256(local_path)

        result["local_size_bytes"] = local_path.stat().st_size
        result["local_sha256"] = local_checksum
        result["size_matches"] = (
            local_path.stat().st_size == response["ContentLength"]
        )
        result["checksum_matches"] = (
            local_checksum == metadata.get("sha256")
        )

    return result
'''

R2_MODULE.write_text(module_code, encoding="utf-8")

# Verify syntax and importability
import py_compile
import sys

py_compile.compile(str(R2_MODULE), doraise=True)

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sales_pipeline.r2_client import (
    create_r2_client,
    create_r2_client_from_colab,
    verify_bucket,
    list_objects,
    calculate_sha256,
    download_file,
    upload_parquet,
    verify_object,
)

print("Module created:", R2_MODULE)
print("Syntax compilation: PASS")
print("Module import: PASS")
print("Functions exported: 8")


Module created: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/r2_client.py
Syntax compilation: PASS
Module import: PASS
Functions exported: 8


In [38]:

from pathlib import Path
import py_compile
import importlib
import sys

PACKAGE_DIR = PROJECT_ROOT / "src" / "sales_pipeline"
R2_MODULE = PACKAGE_DIR / "r2_client.py"

source = R2_MODULE.read_text(encoding="utf-8")

start_marker = "def create_r2_client_from_colab():"
end_marker = "def verify_bucket("

start = source.index(start_marker)
end = source.index(end_marker, start)

replacement = '''def create_r2_client_from_colab():
    """Load R2 credentials using the existing Colab Secret names."""
    try:
        from google.colab import userdata
    except ImportError as exc:
        raise RuntimeError(
            "Colab Secrets are unavailable. Use create_r2_client() "
            "with explicitly supplied credentials."
        ) from exc

    account_id = userdata.get("R2_ACCOUNT_ID").strip()
    access_key_id = userdata.get("R2_ACCESS_KEY").strip()
    secret_access_key = userdata.get("R2_SECRET_ACCESS_KEY").strip()

    return create_r2_client(
        account_id,
        access_key_id,
        secret_access_key,
    )


def get_bucket_name_from_colab():
    """Load the R2 bucket name using the existing Colab Secret name."""
    from google.colab import userdata

    return userdata.get("R2_BUCKET-NAME").strip()


'''

updated_source = source[:start] + replacement + source[end:]
R2_MODULE.write_text(updated_source, encoding="utf-8")

# Compile and reload the updated module
py_compile.compile(str(R2_MODULE), doraise=True)

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import sales_pipeline.r2_client as r2_module
importlib.reload(r2_module)

# Load configuration without displaying credentials
BUCKET_NAME = r2_module.get_bucket_name_from_colab()
r2 = r2_module.create_r2_client_from_colab()

print("Updated module compilation: PASS")
print("Module reload: PASS")
print("R2 client initialization: PASS")
print("Bucket name loaded:", BUCKET_NAME)


Updated module compilation: PASS
Module reload: PASS
R2 client initialization: PASS
Bucket name loaded: sales-analytics-pipeline


In [39]:
# TEST R2 client connection and bucket acess
from sales_pipeline.r2_client import (
    create_r2_client_from_colab,
    verify_bucket,
    list_objects,
)

# Initialize client using Colab Secrets
r2 = create_r2_client_from_colab()

# Verify existing bucket
bucket_verified = verify_bucket(
    r2,
    BUCKET_NAME
)

print("R2 client initialization: PASS")
print("Bucket verification:", "PASS" if bucket_verified else "FAIL")

# List existing Bronze orders objects
sample_objects = list(
    list_objects(
        r2,
        BUCKET_NAME,
        prefix="bronze/olist/olist_orders_dataset/"
    )
)

parquet_objects = [
    obj for obj in sample_objects
    if obj["Key"].endswith(".parquet")
]

print("\nBronze orders Parquet objects:", len(parquet_objects))

for obj in parquet_objects[:5]:
    print(
        obj["Key"],
        f"({obj['Size'] / (1024**2):.2f} MB)"
    )

assert bucket_verified
assert len(parquet_objects) > 0

print("\nR2 CLIENT INTEGRATION: PASS")

R2 client initialization: PASS
Bucket verification: PASS

Bronze orders Parquet objects: 2
bronze/olist/olist_orders_dataset/part-00000-9755c64d-8b52-468b-b3af-0e534639f06c-c000.snappy.parquet (6.36 MB)
bronze/olist/olist_orders_dataset/part-00001-9755c64d-8b52-468b-b3af-0e534639f06c-c000.snappy.parquet (3.93 MB)

R2 CLIENT INTEGRATION: PASS


In [40]:
#test download and checksum verification
from pathlib import Path

from sales_pipeline.r2_client import (
    download_file,
    verify_object,
    calculate_sha256,
)

# Select an existing Bronze object
SOURCE_KEY = parquet_objects[0]["Key"]

# Temporary download location
DOWNLOAD_DIR = Path("/content/phase7_module_test")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_FILE = DOWNLOAD_DIR / Path(SOURCE_KEY).name

# Download through reusable module
downloaded_path = download_file(
    r2,
    BUCKET_NAME,
    SOURCE_KEY,
    LOCAL_FILE,
)

print("Download: PASS")
print("Downloaded file:", downloaded_path)
print(
    "File size:",
    round(downloaded_path.stat().st_size / (1024**2), 3),
    "MB"
)

# Calculate local checksum
local_checksum = calculate_sha256(downloaded_path)

print("\nLocal SHA-256:", local_checksum)

# Inspect remote metadata
verification = verify_object(
    r2,
    BUCKET_NAME,
    SOURCE_KEY,
    downloaded_path,
)

print("\nVerification results:")
for key, value in verification.items():
    print(f"{key}: {value}")

assert verification["exists"]
assert verification["size_matches"]

print("\nDOWNLOAD AND SIZE VERIFICATION: PASS")

Download: PASS
Downloaded file: /content/phase7_module_test/part-00000-9755c64d-8b52-468b-b3af-0e534639f06c-c000.snappy.parquet
File size: 6.357 MB

Local SHA-256: dfababa80bbdb139b8e3348d96074f6ef13048ad8f5840675e43a5fa0384935f

Verification results:
object_key: bronze/olist/olist_orders_dataset/part-00000-9755c64d-8b52-468b-b3af-0e534639f06c-c000.snappy.parquet
remote_size_bytes: 6665407
remote_sha256_metadata: dfababa80bbdb139b8e3348d96074f6ef13048ad8f5840675e43a5fa0384935f
exists: True
local_size_bytes: 6665407
local_sha256: dfababa80bbdb139b8e3348d96074f6ef13048ad8f5840675e43a5fa0384935f
size_matches: True
checksum_matches: True

DOWNLOAD AND SIZE VERIFICATION: PASS


In [41]:
import importlib
from sales_pipeline import r2_client

importlib.reload(r2_client)

client = r2_client.create_r2_client_from_colab()
bucket = r2_client.get_bucket_name_from_colab()

assert LOCAL_OUTPUT.exists(), "Transformed Parquet file not found."

module_object_key = (
    "phase7-test/module-upload/" + LOCAL_OUTPUT.name
)

# Upload using reusable module
upload_result = r2_client.upload_parquet(
    client=client,
    bucket_name=bucket,
    local_path=LOCAL_OUTPUT,
    object_key=module_object_key,
    overwrite=False
)

print("UPLOAD RESULT:", upload_result)

# Verify uploaded object
verification = r2_client.verify_object(
    client=client,
    bucket_name=bucket,
    object_key=module_object_key,
    local_path=LOCAL_OUTPUT
)

print("\nVERIFICATION RESULT:")
for key, value in verification.items():
    print(f"{key}: {value}")

assert verification["size_matches"]
assert verification["checksum_matches"]

print("\nREUSABLE MODULE UPLOAD TEST: PASS")

UPLOAD RESULT: {'object_key': 'phase7-test/module-upload/part-00000-4a32f856-e808-4f3e-b6a6-6e36f47a68a5-c000.snappy.parquet', 'size_bytes': 6585047, 'sha256': 'de8ed820e8af909590d613fef84a591624e4e625e897e540481bdd728c77bbd8'}

VERIFICATION RESULT:
object_key: phase7-test/module-upload/part-00000-4a32f856-e808-4f3e-b6a6-6e36f47a68a5-c000.snappy.parquet
remote_size_bytes: 6585047
remote_sha256_metadata: de8ed820e8af909590d613fef84a591624e4e625e897e540481bdd728c77bbd8
exists: True
local_size_bytes: 6585047
local_sha256: de8ed820e8af909590d613fef84a591624e4e625e897e540481bdd728c77bbd8
size_matches: True
checksum_matches: True

REUSABLE MODULE UPLOAD TEST: PASS


In [42]:
#Add PySpark integration helpers
import inspect
from sales_pipeline import r2_client

print("Existing PySpark integration helpers:")

for function_name in [
    "read_parquet_from_r2",
    "write_spark_parquet_to_r2"
]:
    function = getattr(r2_client, function_name, None)

    if function is None:
        print(f"{function_name}: NOT IMPLEMENTED")
    else:
        print(f"{function_name}: ALREADY IMPLEMENTED")
        print(inspect.signature(function))

Existing PySpark integration helpers:
read_parquet_from_r2: NOT IMPLEMENTED
write_spark_parquet_to_r2: NOT IMPLEMENTED


In [43]:
#Implement reusable PySpark helpers
from pathlib import Path

R2_MODULE = SRC_DIR / "sales_pipeline" / "r2_client.py"

new_helpers = '''

def read_parquet_from_r2(
    spark,
    client,
    bucket_name: str,
    object_key: str,
    local_dir: str | Path,
):
    """Download one R2 Parquet object and return it as a Spark DataFrame."""
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)

    local_path = local_dir / Path(object_key).name

    download_file(
        client=client,
        bucket_name=bucket_name,
        object_key=object_key,
        destination=local_path,
    )

    dataframe = spark.read.parquet(str(local_path))
    return dataframe


def write_spark_parquet_to_r2(
    dataframe,
    client,
    bucket_name: str,
    prefix: str,
    local_dir: str | Path,
    mode: str = "error",
):
    """
    Write a Spark DataFrame as Parquet and upload its part files to R2.

    mode='error' prevents overwriting existing R2 objects.
    mode='overwrite' explicitly permits replacement of matching object keys.
    """
    if mode not in ("error", "overwrite"):
        raise ValueError("mode must be 'error' or 'overwrite'.")

    local_dir = Path(local_dir)

    if local_dir.exists():
        import shutil
        shutil.rmtree(local_dir)

    local_dir.mkdir(parents=True, exist_ok=True)

    dataframe.write.mode("overwrite").parquet(str(local_dir))

    parquet_files = sorted(local_dir.glob("*.parquet"))

    if not parquet_files:
        raise RuntimeError("Spark did not produce any Parquet part files.")

    prefix = prefix.strip("/")
    uploaded_objects = []

    for parquet_file in parquet_files:
        object_key = f"{prefix}/{parquet_file.name}" if prefix else parquet_file.name

        result = upload_parquet(
            client=client,
            bucket_name=bucket_name,
            local_path=parquet_file,
            object_key=object_key,
            overwrite=(mode == "overwrite"),
        )

        uploaded_objects.append(result)

    return {
        "uploaded_file_count": len(uploaded_objects),
        "uploaded_objects": uploaded_objects,
        "total_size_bytes": sum(
            item["size_bytes"] for item in uploaded_objects
        ),
    }
'''

existing_source = R2_MODULE.read_text(encoding="utf-8")

assert "def read_parquet_from_r2(" not in existing_source
assert "def write_spark_parquet_to_r2(" not in existing_source

with R2_MODULE.open("a", encoding="utf-8") as file:
    file.write(new_helpers)

print("PySpark helpers added successfully.")
print("Updated module:", R2_MODULE)

PySpark helpers added successfully.
Updated module: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/r2_client.py


In [45]:
import importlib
from sales_pipeline import r2_client

# Reload updated module
importlib.reload(r2_client)

# Existing R2 configuration
client = r2_client.create_r2_client_from_colab()
bucket = r2_client.get_bucket_name_from_colab()

# Restore the Bronze object key
R2_INPUT_KEY = (
    "bronze/olist/olist_orders_dataset/"
    "part-00000-9755c64d-8b52-468b-b3af-0e534639f06c-c000.snappy.parquet"
)

# Read R2 Parquet using the reusable helper
r2_spark_df = r2_client.read_parquet_from_r2(
    spark=spark,
    client=client,
    bucket_name=bucket,
    object_key=R2_INPUT_KEY,
    local_dir="/content/phase7_r2_helper_read"
)

print("R2-to-PySpark helper: SUCCESS")
print("Rows:", r2_spark_df.count())
print("Columns:", len(r2_spark_df.columns))

r2_spark_df.printSchema()
r2_spark_df.show(5, truncate=False)

assert r2_spark_df.count() == 61530
assert len(r2_spark_df.columns) == 8

print("READ HELPER TEST: PASS")

R2-to-PySpark helper: SUCCESS
Rows: 61530
Columns: 8
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+--

In [46]:
#Test write_spark_parquet_to_r2()
import importlib
from pyspark.sql import functions as F
from sales_pipeline import r2_client

importlib.reload(r2_client)

# Transform data using Spark
test_output_df = (
    r2_spark_df
    .filter(F.col("order_status") == "delivered")
    .withColumn(
        "purchase_year",
        F.year("order_purchase_timestamp")
    )
    .withColumn(
        "purchase_month",
        F.month("order_purchase_timestamp")
    )
)

# Upload through reusable PySpark-to-R2 helper
write_result = r2_client.write_spark_parquet_to_r2(
    dataframe=test_output_df,
    client=client,
    bucket_name=bucket,
    prefix="phase7-test/spark-helper-output/orders",
    local_dir="/content/phase7_spark_helper_output",
    mode="error"
)

print("SPARK-TO-R2 WRITE RESULT")
print(write_result)

assert write_result["uploaded_file_count"] >= 1
assert write_result["total_size_bytes"] > 0

print("\nSPARK-TO-R2 WRITE HELPER: PASS")

SPARK-TO-R2 WRITE RESULT
{'uploaded_file_count': 1, 'uploaded_objects': [{'object_key': 'phase7-test/spark-helper-output/orders/part-00000-3829d190-6529-4ebd-a09e-83afb63473e5-c000.snappy.parquet', 'size_bytes': 6531649, 'sha256': 'bf53096d4d019b08fd3cfb6adc28ce0b09237b0d51eed63decf755d9bde0766d'}], 'total_size_bytes': 6531649}

SPARK-TO-R2 WRITE HELPER: PASS


In [47]:
from pyspark.sql import functions as F
from sales_pipeline import r2_client

# Get uploaded object key from previous write result
uploaded_key = write_result["uploaded_objects"][0]["object_key"]

# Read uploaded data back through reusable helper
recovered_spark_df = r2_client.read_parquet_from_r2(
    spark=spark,
    client=client,
    bucket_name=bucket,
    object_key=uploaded_key,
    local_dir="/content/phase7_spark_helper_recovered"
)

# Compare both datasets
source_count = test_output_df.count()
recovered_count = recovered_spark_df.count()

source_only = test_output_df.exceptAll(recovered_spark_df).count()
recovered_only = recovered_spark_df.exceptAll(test_output_df).count()

print("SOURCE ROWS:", source_count)
print("RECOVERED ROWS:", recovered_count)
print("SOURCE-ONLY ROWS:", source_only)
print("RECOVERED-ONLY ROWS:", recovered_only)

assert source_count == recovered_count
assert source_only == 0
assert recovered_only == 0

print("\nREUSABLE PYSPARK + R2 ROUND-TRIP: PASS")

SOURCE ROWS: 59702
RECOVERED ROWS: 59702
SOURCE-ONLY ROWS: 0
RECOVERED-ONLY ROWS: 0

REUSABLE PYSPARK + R2 ROUND-TRIP: PASS


In [48]:
#Test multiple Parquet part
from sales_pipeline import r2_client

# Force Spark to generate 3 output partitions
multi_partition_df = test_output_df.repartition(3)

multi_write_result = r2_client.write_spark_parquet_to_r2(
    dataframe=multi_partition_df,
    client=client,
    bucket_name=bucket,
    prefix="phase7-test/multi-part-output/orders",
    local_dir="/content/phase7_multi_part_output",
    mode="error"
)

print("Uploaded Parquet files:", multi_write_result["uploaded_file_count"])
print("Total uploaded size:", multi_write_result["total_size_bytes"])

for item in multi_write_result["uploaded_objects"]:
    print(item["object_key"], item["size_bytes"])

assert multi_write_result["uploaded_file_count"] == 3

print("\nMULTI-PART PARQUET UPLOAD: PASS")

Uploaded Parquet files: 3
Total uploaded size: 6578929
phase7-test/multi-part-output/orders/part-00000-3b490781-bb61-483f-842d-ce51093ee657-c000.snappy.parquet 2192732
phase7-test/multi-part-output/orders/part-00001-3b490781-bb61-483f-842d-ce51093ee657-c000.snappy.parquet 2193008
phase7-test/multi-part-output/orders/part-00002-3b490781-bb61-483f-842d-ce51093ee657-c000.snappy.parquet 2193189

MULTI-PART PARQUET UPLOAD: PASS


In [49]:
#Read all three R2 objects and reconcile
from pathlib import Path
from sales_pipeline import r2_client

# Download all three objects into one local directory
MULTI_LOCAL_DIR = Path("/content/phase7_multi_part_recovered")
MULTI_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

downloaded_files = []

for item in multi_write_result["uploaded_objects"]:
    object_key = item["object_key"]
    local_path = MULTI_LOCAL_DIR / Path(object_key).name

    r2_client.download_file(
        client=client,
        bucket_name=bucket,
        object_key=object_key,
        destination=local_path
    )

    downloaded_files.append(local_path)

# Read all downloaded Parquet parts together
multi_recovered_df = spark.read.parquet(
    str(MULTI_LOCAL_DIR)
)

# Reconcile against source DataFrame
source_count = test_output_df.count()
recovered_count = multi_recovered_df.count()

source_only = test_output_df.exceptAll(
    multi_recovered_df
).count()

recovered_only = multi_recovered_df.exceptAll(
    test_output_df
).count()

print("Downloaded files:", len(downloaded_files))
print("Source rows:", source_count)
print("Recovered rows:", recovered_count)
print("Source-only rows:", source_only)
print("Recovered-only rows:", recovered_only)

assert len(downloaded_files) == 3
assert source_count == recovered_count
assert source_only == 0
assert recovered_only == 0

print("\nMULTI-PART R2 + PYSPARK RECONCILIATION: PASS")

Downloaded files: 3
Source rows: 59702
Recovered rows: 59702
Source-only rows: 0
Recovered-only rows: 0

MULTI-PART R2 + PYSPARK RECONCILIATION: PASS


In [50]:
import inspect
from sales_pipeline import r2_client

print(inspect.getsource(
    r2_client.write_spark_parquet_to_r2
))

def write_spark_parquet_to_r2(
    dataframe,
    client,
    bucket_name: str,
    prefix: str,
    local_dir: str | Path,
    mode: str = "error",
):
    """
    Write a Spark DataFrame as Parquet and upload its part files to R2.

    mode='error' prevents overwriting existing R2 objects.
    mode='overwrite' explicitly permits replacement of matching object keys.
    """
    if mode not in ("error", "overwrite"):
        raise ValueError("mode must be 'error' or 'overwrite'.")

    local_dir = Path(local_dir)

    if local_dir.exists():
        import shutil
        shutil.rmtree(local_dir)

    local_dir.mkdir(parents=True, exist_ok=True)

    dataframe.write.mode("overwrite").parquet(str(local_dir))

    parquet_files = sorted(local_dir.glob("*.parquet"))

    if not parquet_files:
        raise RuntimeError("Spark did not produce any Parquet part files.")

    prefix = prefix.strip("/")
    uploaded_objects = []

    for parquet_file in parquet_files:
        object_key = f"{pref

In [51]:
from pathlib import Path

R2_MODULE = SRC_DIR / "sales_pipeline" / "r2_client.py"

source = R2_MODULE.read_text(encoding="utf-8")

start_marker = "def write_spark_parquet_to_r2("
start = source.index(start_marker)

# Locate the next top-level function, if present
next_function = source.find("\ndef ", start + len(start_marker))

if next_function == -1:
    end = len(source)
else:
    end = next_function + 1

new_function = '''
def write_spark_parquet_to_r2(
    dataframe,
    client,
    bucket_name: str,
    prefix: str,
    local_dir: str | Path,
    mode: str = "error",
):
    """
    Write Spark output as Parquet and upload all part files to R2.

    mode='error':
        Fail if the destination prefix already contains objects.

    mode='overwrite':
        Upload and verify the new output first, then remove stale
        objects from the destination prefix.

    Failed uploads trigger cleanup of objects created by this run.
    """
    import tempfile

    if mode not in ("error", "overwrite"):
        raise ValueError("mode must be 'error' or 'overwrite'.")

    prefix = prefix.strip("/")

    if not prefix:
        raise ValueError(
            "A non-empty destination prefix is required."
        )

    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)

    # Inspect existing destination objects before processing.
    existing_objects = list(
        list_objects(client, bucket_name, prefix=prefix + "/")
    )

    if mode == "error" and existing_objects:
        raise FileExistsError(
            f"Destination prefix already contains objects: {prefix}"
        )

    uploaded_objects = []
    uploaded_keys = []

    try:
        # Use an isolated temporary directory. Do not erase local_dir.
        with tempfile.TemporaryDirectory(
            prefix="spark-r2-",
            dir=str(local_dir),
        ) as temporary_directory:

            output_path = Path(temporary_directory) / "parquet_output"

            dataframe.write.mode("error").parquet(
                str(output_path)
            )

            parquet_files = sorted(
                output_path.glob("*.parquet")
            )

            if not parquet_files:
                raise RuntimeError(
                    "Spark did not produce any Parquet part files."
                )

            # Upload and verify every new part before deleting old data.
            for parquet_file in parquet_files:
                object_key = (
                    f"{prefix}/{parquet_file.name}"
                )

                result = upload_parquet(
                    client=client,
                    bucket_name=bucket_name,
                    local_path=parquet_file,
                    object_key=object_key,
                    overwrite=False,
                )

                verification = verify_object(
                    client=client,
                    bucket_name=bucket_name,
                    object_key=object_key,
                    local_path=parquet_file,
                )

                if not (
                    verification["size_matches"]
                    and verification["checksum_matches"]
                ):
                    raise RuntimeError(
                        f"Upload verification failed: {object_key}"
                    )

                uploaded_objects.append(result)
                uploaded_keys.append(object_key)

        # Remove stale objects only after all new uploads are verified.
        if mode == "overwrite":
            stale_keys = [
                item["Key"]
                for item in existing_objects
                if item["Key"] not in uploaded_keys
            ]

            # S3-compatible DeleteObjects supports batches of 1000.
            for offset in range(0, len(stale_keys), 1000):
                batch = stale_keys[offset:offset + 1000]

                response = client.delete_objects(
                    Bucket=bucket_name,
                    Delete={
                        "Objects": [
                            {"Key": key} for key in batch
                        ],
                        "Quiet": False,
                    },
                )

                errors = response.get("Errors", [])

                if errors:
                    raise RuntimeError(
                        f"Failed to delete stale R2 objects: {errors}"
                    )

    except Exception:
        # Roll back only objects uploaded by this execution.
        if uploaded_keys:
            for offset in range(0, len(uploaded_keys), 1000):
                batch = uploaded_keys[offset:offset + 1000]

                client.delete_objects(
                    Bucket=bucket_name,
                    Delete={
                        "Objects": [
                            {"Key": key} for key in batch
                        ],
                        "Quiet": True,
                    },
                )

        raise

    return {
        "uploaded_file_count": len(uploaded_objects),
        "uploaded_objects": uploaded_objects,
        "total_size_bytes": sum(
            item["size_bytes"] for item in uploaded_objects
        ),
        "destination_prefix": prefix,
        "mode": mode,
        "verification": "PASS",
    }
'''

updated_source = (
    source[:start] + new_function + "\n\n" + source[end:]
)

R2_MODULE.write_text(updated_source, encoding="utf-8")

print("Function replacement: SUCCESS")
print("Updated:", R2_MODULE)

Function replacement: SUCCESS
Updated: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/r2_client.py


In [52]:
import importlib
from sales_pipeline import r2_client

importlib.reload(r2_client)

# Generate two output partitions
overwrite_test_df = test_output_df.repartition(2)

# Replace the existing test prefix
overwrite_result = r2_client.write_spark_parquet_to_r2(
    dataframe=overwrite_test_df,
    client=client,
    bucket_name=bucket,
    prefix="phase7-test/multi-part-output/orders",
    local_dir="/content/phase7_overwrite_test",
    mode="overwrite"
)

# Inspect the destination prefix after replacement
remaining_objects = list(
    r2_client.list_objects(
        client,
        bucket,
        prefix="phase7-test/multi-part-output/orders/"
    )
)

print("OVERWRITE RESULT")
print(overwrite_result)

print("\nObjects remaining in destination:", len(remaining_objects))

for obj in remaining_objects:
    print(obj["Key"], obj["Size"])

assert overwrite_result["uploaded_file_count"] == 2
assert len(remaining_objects) == 2
assert overwrite_result["verification"] == "PASS"

print("\nOVERWRITE PREFIX REPLACEMENT: PASS")

OVERWRITE RESULT
{'uploaded_file_count': 2, 'uploaded_objects': [{'object_key': 'phase7-test/multi-part-output/orders/part-00000-13fdbf1a-d236-4b99-b69b-47087a80d895-c000.snappy.parquet', 'size_bytes': 3279665, 'sha256': 'a5a9de081e9dc9a8b6a2365f9a7df86fbba0b873a0d1de4542a068ef87a27d6a'}, {'object_key': 'phase7-test/multi-part-output/orders/part-00001-13fdbf1a-d236-4b99-b69b-47087a80d895-c000.snappy.parquet', 'size_bytes': 3278667, 'sha256': '82490c201391710759c4b6cb57951300a81837001fd6c458bf6f3afbacc21c8a'}], 'total_size_bytes': 6558332, 'destination_prefix': 'phase7-test/multi-part-output/orders', 'mode': 'overwrite', 'verification': 'PASS'}

Objects remaining in destination: 2
phase7-test/multi-part-output/orders/part-00000-13fdbf1a-d236-4b99-b69b-47087a80d895-c000.snappy.parquet 3279665
phase7-test/multi-part-output/orders/part-00001-13fdbf1a-d236-4b99-b69b-47087a80d895-c000.snappy.parquet 3278667

OVERWRITE PREFIX REPLACEMENT: PASS


In [53]:
from pathlib import Path

# Download the two replacement objects
FINAL_RECOVERY_DIR = Path("/content/phase7_final_overwrite_recovery")
FINAL_RECOVERY_DIR.mkdir(parents=True, exist_ok=True)

final_objects = list(
    r2_client.list_objects(
        client,
        bucket,
        prefix="phase7-test/multi-part-output/orders/"
    )
)

for obj in final_objects:
    destination = FINAL_RECOVERY_DIR / Path(obj["Key"]).name

    r2_client.download_file(
        client,
        bucket,
        obj["Key"],
        destination
    )

# Read all replacement Parquet parts
final_recovered_df = spark.read.parquet(
    str(FINAL_RECOVERY_DIR)
)

# Bidirectional reconciliation
source_count = test_output_df.count()
recovered_count = final_recovered_df.count()

source_only = test_output_df.exceptAll(final_recovered_df).count()
recovered_only = final_recovered_df.exceptAll(test_output_df).count()

print("Replacement files:", len(final_objects))
print("Source rows:", source_count)
print("Recovered rows:", recovered_count)
print("Source-only rows:", source_only)
print("Recovered-only rows:", recovered_only)

assert len(final_objects) == 2
assert source_count == recovered_count == 59702
assert source_only == 0
assert recovered_only == 0

print("\nFINAL OVERWRITE RECONCILIATION: PASS")


Replacement files: 2
Source rows: 59702
Recovered rows: 59702
Source-only rows: 0
Recovered-only rows: 0

FINAL OVERWRITE RECONCILIATION: PASS


#### Automated testing

In [54]:
TESTS_DIR = PROJECT_ROOT / "tests"

print("Tests directory exists:", TESTS_DIR.exists())

if TESTS_DIR.exists():
    for path in sorted(TESTS_DIR.iterdir()):
        print(
            "[DIR] " if path.is_dir() else "[FILE]",
            path.name
        )

Tests directory exists: True
[FILE] __init__.py
[DIR]  __pycache__
[FILE] test_incremental.py
[FILE] test_incremental_integration.py
[FILE] test_phase4_storage_modeling.py
[FILE] test_pipeline.py
[FILE] test_spark_transformations.py


In [55]:
from pathlib import Path

test_files = [
    TESTS_DIR / "test_phase4_storage_modeling.py",
    TESTS_DIR / "test_spark_transformations.py",
    TESTS_DIR / "test_pipeline.py",
]

for file_path in test_files:
    print("\n" + "=" * 75)
    print("FILE:", file_path.name)
    print("=" * 75)

    if file_path.exists():
        print(file_path.read_text(encoding="utf-8"))
    else:
        print("File not found.")


FILE: test_phase4_storage_modeling.py
"""Persistent Phase 4 tests for Parquet storage and dimensional modeling."""

import unittest
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


REPO_ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = REPO_ROOT / "data"
ANALYTICS_DIR = DATA_DIR / "gold" / "analytics"
DIMENSIONAL_DIR = DATA_DIR / "gold" / "dimensional"


class TestPhase4StorageAndModeling(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        required_paths = [
            ANALYTICS_DIR / "monthly_sales_analytics",
            DIMENSIONAL_DIR / "dim_customer",
            DIMENSIONAL_DIR / "dim_product",
            DIMENSIONAL_DIR / "dim_seller",
            DIMENSIONAL_DIR / "dim_order",
            DIMENSIONAL_DIR / "fact_sales",
        ]

        missing = [str(p) for p in required_paths if not p.exists()]
        if missing:
            raise unittest.SkipTest(
                "Phase 4 Parquet data unavai

In [56]:
#Create test_r2_client.py
from pathlib import Path

TEST_FILE = TESTS_DIR / "test_r2_client.py"

test_code = r'''
"""Unit tests for Cloudflare R2 storage integration."""

import sys
import tempfile
import unittest
import hashlib

from pathlib import Path
from unittest.mock import Mock, patch

from botocore.exceptions import ClientError

REPO_ROOT = Path(__file__).resolve().parents[1]
SRC_DIR = REPO_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from sales_pipeline import r2_client


class TestR2Client(unittest.TestCase):

    def test_create_r2_client(self):
        with patch.object(r2_client.boto3, "client") as mock_client:
            r2_client.create_r2_client(
                "account123",
                "access123",
                "secret123"
            )

            mock_client.assert_called_once()

    def test_empty_credentials_raise_error(self):
        with self.assertRaises(ValueError):
            r2_client.create_r2_client("", "access", "secret")

    def test_calculate_sha256(self):
        with tempfile.TemporaryDirectory() as temp_dir:
            path = Path(temp_dir) / "sample.txt"
            path.write_bytes(b"sales pipeline")

            expected = hashlib.sha256(
                b"sales pipeline"
            ).hexdigest()

            self.assertEqual(
                r2_client.calculate_sha256(path),
                expected
            )

    def test_upload_parquet_success(self):
        with tempfile.TemporaryDirectory() as temp_dir:
            path = Path(temp_dir) / "sample.parquet"
            path.write_bytes(b"parquet test data")

            client = Mock()
            client.head_object.side_effect = ClientError(
                {
                    "Error": {"Code": "404"},
                    "ResponseMetadata": {"HTTPStatusCode": 404}
                },
                "HeadObject"
            )

            result = r2_client.upload_parquet(
                client,
                "test-bucket",
                path,
                "test/sample.parquet"
            )

            self.assertEqual(
                result["size_bytes"],
                path.stat().st_size
            )
            self.assertEqual(
                result["sha256"],
                r2_client.calculate_sha256(path)
            )
            client.upload_file.assert_called_once()

    def test_upload_rejects_existing_object(self):
        with tempfile.TemporaryDirectory() as temp_dir:
            path = Path(temp_dir) / "sample.parquet"
            path.write_bytes(b"test")

            client = Mock()
            client.head_object.return_value = {
                "ContentLength": 4
            }

            with self.assertRaises(FileExistsError):
                r2_client.upload_parquet(
                    client,
                    "test-bucket",
                    path,
                    "existing/sample.parquet",
                    overwrite=False
                )

            client.upload_file.assert_not_called()

    def test_upload_rejects_non_parquet_file(self):
        with tempfile.TemporaryDirectory() as temp_dir:
            path = Path(temp_dir) / "sample.csv"
            path.write_text("a,b")

            with self.assertRaises(ValueError):
                r2_client.upload_parquet(
                    Mock(),
                    "test-bucket",
                    path,
                    "sample.csv"
                )

    def test_download_file(self):
        with tempfile.TemporaryDirectory() as temp_dir:
            destination = Path(temp_dir) / "download.parquet"

            def mock_download(bucket, key, filename):
                Path(filename).write_bytes(b"downloaded data")

            client = Mock()
            client.download_file.side_effect = mock_download

            result = r2_client.download_file(
                client,
                "test-bucket",
                "source/sample.parquet",
                destination
            )

            self.assertEqual(result, destination)
            self.assertTrue(destination.exists())
            self.assertEqual(
                destination.read_bytes(),
                b"downloaded data"
            )

    def test_verify_object_checksum(self):
        with tempfile.TemporaryDirectory() as temp_dir:
            path = Path(temp_dir) / "sample.parquet"
            path.write_bytes(b"verified data")

            checksum = r2_client.calculate_sha256(path)

            client = Mock()
            client.head_object.return_value = {
                "ContentLength": path.stat().st_size,
                "Metadata": {"sha256": checksum}
            }

            result = r2_client.verify_object(
                client,
                "test-bucket",
                "sample.parquet",
                path
            )

            self.assertTrue(result["exists"])
            self.assertTrue(result["size_matches"])
            self.assertTrue(result["checksum_matches"])


if __name__ == "__main__":
    unittest.main()
'''

TEST_FILE.write_text(test_code.strip() + "\n", encoding="utf-8")

print("Test file created:", TEST_FILE)
print("Total test methods:", test_code.count("    def test_"))

Test file created: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/tests/test_r2_client.py
Total test methods: 8


In [57]:
#add Spark–R2 helper tests
import sys
import unittest

sys.path.insert(0, str(SRC_DIR))

suite = unittest.defaultTestLoader.discover(
    start_dir=str(TESTS_DIR),
    pattern="test_r2_client.py"
)

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

print("\nR2 CLIENT UNIT TEST SUMMARY")
print("Tests run:", result.testsRun)
print("Failures:", len(result.failures))
print("Errors:", len(result.errors))

assert result.wasSuccessful(), "Some R2 client tests failed."

test_calculate_sha256 (test_r2_client.TestR2Client.test_calculate_sha256) ... ok
test_create_r2_client (test_r2_client.TestR2Client.test_create_r2_client) ... ok
test_download_file (test_r2_client.TestR2Client.test_download_file) ... ok
test_empty_credentials_raise_error (test_r2_client.TestR2Client.test_empty_credentials_raise_error) ... ok
test_upload_parquet_success (test_r2_client.TestR2Client.test_upload_parquet_success) ... ok
test_upload_rejects_existing_object (test_r2_client.TestR2Client.test_upload_rejects_existing_object) ... ok
test_upload_rejects_non_parquet_file (test_r2_client.TestR2Client.test_upload_rejects_non_parquet_file) ... ok
test_verify_object_checksum (test_r2_client.TestR2Client.test_verify_object_checksum) ... ok

----------------------------------------------------------------------
Ran 8 tests in 0.141s

OK



R2 CLIENT UNIT TEST SUMMARY
Tests run: 8
Failures: 0
Errors: 0


In [59]:
#add Spark–R2 helper tests
from pathlib import Path

test_file = TESTS_DIR / "test_r2_client.py"

additional_tests = r'''

class TestR2SparkHelpers(unittest.TestCase):

    def test_read_parquet_from_r2(self):
        from unittest.mock import MagicMock, patch
        from sales_pipeline.r2_client import read_parquet_from_r2

        spark = MagicMock()
        spark.read.parquet.return_value = "mock_dataframe"

        with patch(
            "sales_pipeline.r2_client.download_file"
        ) as mock_download:
            result = read_parquet_from_r2(
                spark=spark,
                client=MagicMock(),
                bucket_name="test-bucket",
                object_key="bronze/orders/part-000.parquet",
                local_dir=Path(self._testMethodName)
            )

        self.assertEqual(result, "mock_dataframe")
        mock_download.assert_called_once()
        spark.read.parquet.assert_called_once()

    def test_write_spark_parquet_to_r2_uploads_parts(self):
        from unittest.mock import MagicMock, patch
        from sales_pipeline.r2_client import write_spark_parquet_to_r2

        dataframe = MagicMock()
        writer = dataframe.write
        writer.mode.return_value = writer

        def fake_parquet(path):
            output = Path(path)
            output.mkdir(parents=True, exist_ok=True)
            (output / "part-00000.parquet").write_bytes(b"part-one")
            (output / "part-00001.parquet").write_bytes(b"part-two")

        writer.parquet.side_effect = fake_parquet

        client = MagicMock()

        with patch(
            "sales_pipeline.r2_client.list_objects",
            return_value=[]
        ), patch(
            "sales_pipeline.r2_client.upload_parquet",
            side_effect=lambda client, bucket_name, local_path, object_key,
            overwrite=False: {
                "object_key": object_key,
                "size": Path(local_path).stat().st_size,
                "sha256": "mock-checksum"
            }
        ), patch(
            "sales_pipeline.r2_client.verify_object",
            return_value={"verified": True}
        ):
            import tempfile

            with tempfile.TemporaryDirectory() as temp_dir:
                result = write_spark_parquet_to_r2(
                    dataframe=dataframe,
                    client=client,
                    bucket_name="test-bucket",
                    prefix="phase7-test/unit-test/",
                    local_dir=Path(temp_dir),
                    mode="error"
                )

        self.assertEqual(result["uploaded_files"], 2)
        self.assertEqual(result["total_size"], len(b"part-one") + len(b"part-two"))
        self.assertTrue(result["verification"] == "PASS")

    def test_write_rejects_existing_prefix_in_error_mode(self):
        from unittest.mock import MagicMock, patch
        from sales_pipeline.r2_client import write_spark_parquet_to_r2

        with patch(
            "sales_pipeline.r2_client.list_objects",
            return_value=[{"Key": "phase7-test/existing/file.parquet"}]
        ):
            with self.assertRaises(Exception):
                write_spark_parquet_to_r2(
                    dataframe=MagicMock(),
                    client=MagicMock(),
                    bucket_name="test-bucket",
                    prefix="phase7-test/existing/",
                    local_dir=Path(self._testMethodName),
                    mode="error"
                )

    def test_write_rejects_empty_prefix(self):
        from unittest.mock import MagicMock
        from sales_pipeline.r2_client import write_spark_parquet_to_r2

        with self.assertRaises(ValueError):
            write_spark_parquet_to_r2(
                dataframe=MagicMock(),
                client=MagicMock(),
                bucket_name="test-bucket",
                prefix="",
                local_dir=Path(self._testMethodName),
                mode="error"
            )

    def test_write_rejects_invalid_mode(self):
        from unittest.mock import MagicMock
        from sales_pipeline.r2_client import write_spark_parquet_to_r2

        with self.assertRaises(ValueError):
            write_spark_parquet_to_r2(
                dataframe=MagicMock(),
                client=MagicMock(),
                bucket_name="test-bucket",
                prefix="phase7-test/invalid-mode/",
                local_dir=Path(self._testMethodName),
                mode="invalid"
            )
'''

with test_file.open("a", encoding="utf-8") as f:
    f.write(additional_tests)

print(f"Added Spark–R2 tests to: {test_file}")

Added Spark–R2 tests to: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/tests/test_r2_client.py


In [60]:
#run the complete R2 test suite
import sys
import unittest

sys.path.insert(0, str(SRC_DIR))

suite = unittest.defaultTestLoader.discover(
    start_dir=str(TESTS_DIR),
    pattern="test_r2_client.py"
)

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

print("\nR2 CLIENT TEST SUMMARY")
print("Tests run:", result.testsRun)
print("Failures:", len(result.failures))
print("Errors:", len(result.errors))
print("Skipped:", len(result.skipped))

assert result.wasSuccessful(), "Some R2 client tests failed."

test_calculate_sha256 (test_r2_client.TestR2Client.test_calculate_sha256) ... ok
test_create_r2_client (test_r2_client.TestR2Client.test_create_r2_client) ... ok
test_download_file (test_r2_client.TestR2Client.test_download_file) ... ok
test_empty_credentials_raise_error (test_r2_client.TestR2Client.test_empty_credentials_raise_error) ... ok
test_upload_parquet_success (test_r2_client.TestR2Client.test_upload_parquet_success) ... ok
test_upload_rejects_existing_object (test_r2_client.TestR2Client.test_upload_rejects_existing_object) ... ok
test_upload_rejects_non_parquet_file (test_r2_client.TestR2Client.test_upload_rejects_non_parquet_file) ... ok
test_verify_object_checksum (test_r2_client.TestR2Client.test_verify_object_checksum) ... ok

----------------------------------------------------------------------
Ran 8 tests in 0.023s

OK



R2 CLIENT TEST SUMMARY
Tests run: 8
Failures: 0
Errors: 0
Skipped: 0


In [61]:
#inspect test discovery

import ast
from pathlib import Path

test_file = TESTS_DIR / "test_r2_client.py"
source = test_file.read_text(encoding="utf-8")

tree = ast.parse(source)

print("TEST CLASSES AND METHODS FOUND IN FILE")
for node in tree.body:
    if isinstance(node, ast.ClassDef):
        methods = [
            item.name for item in node.body
            if isinstance(item, (ast.FunctionDef, ast.AsyncFunctionDef))
            and item.name.startswith("test_")
        ]
        print(f"\n{node.name}: {len(methods)} tests")
        for method in methods:
            print("  -", method)

print("\nFILE ENDING")
print("\n".join(source.splitlines()[-35:]))

TEST CLASSES AND METHODS FOUND IN FILE

TestR2Client: 8 tests
  - test_create_r2_client
  - test_empty_credentials_raise_error
  - test_calculate_sha256
  - test_upload_parquet_success
  - test_upload_rejects_existing_object
  - test_upload_rejects_non_parquet_file
  - test_download_file
  - test_verify_object_checksum

TestR2SparkHelpers: 5 tests
  - test_read_parquet_from_r2
  - test_write_spark_parquet_to_r2_uploads_parts
  - test_write_rejects_existing_prefix_in_error_mode
  - test_write_rejects_empty_prefix
  - test_write_rejects_invalid_mode

TestR2SparkHelpers: 5 tests
  - test_read_parquet_from_r2
  - test_write_spark_parquet_to_r2_uploads_parts
  - test_write_rejects_existing_prefix_in_error_mode
  - test_write_rejects_empty_prefix
  - test_write_rejects_invalid_mode

FILE ENDING
                    dataframe=MagicMock(),
                    client=MagicMock(),
                    bucket_name="test-bucket",
                    prefix="phase7-test/existing/",
                  

In [71]:
from pathlib import Path

test_file = TESTS_DIR / "test_r2_client.py"
source = test_file.read_text(encoding="utf-8")

old = '"size": Path(local_path).stat().st_size,'
new = '"size_bytes": Path(local_path).stat().st_size,'

count = source.count(old)
print("Matching mock fields found:", count)

if count == 0:
    raise ValueError("No matching mock field found. No changes made.")

source = source.replace(old, new)
test_file.write_text(source, encoding="utf-8")

print("Updated", count, "mock return field(s).")

Matching mock fields found: 2
Updated 2 mock return field(s).


In [76]:
import sys
import importlib
import unittest

sys.path.insert(0, str(TESTS_DIR))
sys.path.insert(0, str(SRC_DIR))

# Remove any previously cached test module
sys.modules.pop("test_r2_client", None)

test_module = importlib.import_module("test_r2_client")
test_module = importlib.reload(test_module)

print("Imported test module:", test_module.__file__)

suite = unittest.TestSuite()
suite.addTests(
    unittest.defaultTestLoader.loadTestsFromTestCase(
        test_module.TestR2Client
    )
)
suite.addTests(
    unittest.defaultTestLoader.loadTestsFromTestCase(
        test_module.TestR2SparkHelpers
    )
)

result = unittest.TextTestRunner(verbosity=2).run(suite)

print("\nR2 CLIENT TEST SUMMARY")
print("Tests run:", result.testsRun)
print("Failures:", len(result.failures))
print("Errors:", len(result.errors))
print("Skipped:", len(result.skipped))

assert result.wasSuccessful(), "Some R2 client tests failed."

test_calculate_sha256 (test_r2_client.TestR2Client.test_calculate_sha256) ... ok
test_create_r2_client (test_r2_client.TestR2Client.test_create_r2_client) ... ok
test_download_file (test_r2_client.TestR2Client.test_download_file) ... ok
test_empty_credentials_raise_error (test_r2_client.TestR2Client.test_empty_credentials_raise_error) ... ok
test_upload_parquet_success (test_r2_client.TestR2Client.test_upload_parquet_success) ... ok
test_upload_rejects_existing_object (test_r2_client.TestR2Client.test_upload_rejects_existing_object) ... ok
test_upload_rejects_non_parquet_file (test_r2_client.TestR2Client.test_upload_rejects_non_parquet_file) ... ok
test_verify_object_checksum (test_r2_client.TestR2Client.test_verify_object_checksum) ... ok
test_read_parquet_from_r2 (test_r2_client.TestR2SparkHelpers.test_read_parquet_from_r2) ... ok
test_write_rejects_empty_prefix (test_r2_client.TestR2SparkHelpers.test_write_rejects_empty_prefix) ... ok
test_write_rejects_existing_prefix_in_error_mode

Imported test module: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/tests/test_r2_client.py

R2 CLIENT TEST SUMMARY
Tests run: 13
Failures: 0
Errors: 0
Skipped: 0


#### Integrate the reusable R2 module into the end-to-end data pipelin

In [77]:
#Inspect the existing orchestration code$0
from pathlib import Path
import ast

files_to_inspect = [
    SRC_DIR / "sales_pipeline" / "pipeline.py",
    SRC_DIR / "sales_pipeline" / "r2_client.py",
]

for file_path in files_to_inspect:
    print("\n" + "=" * 75)
    print(f"FILE: {file_path.name}")
    print("=" * 75)

    source = file_path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    print("\nFUNCTION SIGNATURES:")
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            args = [arg.arg for arg in node.args.args]
            print(f"  {node.name}({', '.join(args)})")

    if file_path.name == "pipeline.py":
        print("\nEXISTING PIPELINE IMPLEMENTATION:")
        print(source)
    else:
        print("\nR2 HELPER IMPLEMENTATION (FUNCTIONS ONLY):")
        for node in tree.body:
            if isinstance(node, ast.FunctionDef):
                print(f"\n--- {node.name} ---")
                print("\n".join(source.splitlines()[
                    node.lineno - 1:node.end_lineno
                ]))


FILE: pipeline.py

FUNCTION SIGNATURES:
  run_pipeline(data_dir, output_dir, database_url)

EXISTING PIPELINE IMPLEMENTATION:
"""Orchestrate the sales analytics data pipeline."""

from pathlib import Path
from datetime import datetime, timezone

from . import validate
from . import transform
from . import transform_products
from . import transform_customers
from . import transform_sales
from . import database


def run_pipeline(data_dir, output_dir, database_url=None):
    data_dir = Path(data_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    started_at = datetime.now(timezone.utc)

    print("=" * 64)
    print("SALES ANALYTICS DATA PIPELINE")
    print("=" * 64)

    # Step 1: Validate source data.
    print("\n[1/5] Validating source data...")
    validation_results = validate.validate_data(data_dir)
    validation_passed = validate.print_report(validation_results)

    if not validation_passed:
        raise RuntimeError(
            "Pi

In [78]:
import ast

candidate_files = [
    SRC_DIR / "sales_pipeline" / "spark_transformations.py",
    SRC_DIR / "sales_pipeline" / "transformation.py",
    SRC_DIR / "sales_pipeline" / "incremental.py",
]

for file_path in candidate_files:
    print("\n" + "=" * 65)

    if not file_path.exists():
        print(f"NOT FOUND: {file_path.name}")
        continue

    print(f"FILE: {file_path.name}")
    source = file_path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    print("\nIMPORTS:")
    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            print(ast.unparse(node))

    print("\nFUNCTIONS AND CLASSES:")
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
            print(f"\n{node.name} — line {node.lineno}")

            if isinstance(node, ast.FunctionDef):
                print("  Parameters:", [
                    arg.arg for arg in node.args.args
                ])


FILE: spark_transformations.py

IMPORTS:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

FUNCTIONS AND CLASSES:

build_order_analytics_spark — line 10
  Parameters: ['orders', 'order_items', 'payments', 'customers']

prepare_sales_items_spark — line 100
  Parameters: ['orders', 'order_items', 'products', 'category_translations']

build_product_analytics_spark — line 145
  Parameters: ['sales_items']

build_category_analytics_spark — line 160
  Parameters: ['sales_items']

FILE: transformation.py

IMPORTS:

FUNCTIONS AND CLASSES:

FILE: incremental.py

IMPORTS:
import hashlib
import json
import logging
import psycopg

FUNCTIONS AND CLASSES:

make_batch_id — line 15
  Parameters: ['source_name', 'batch_key']

register_batch — line 34
  Parameters: ['database_url', 'batch_id', 'batch_type']

mark_batch_succeeded — line 126
  Parameters: ['database_url', 'batch_id', 'source_rows', 'inserted_rows', 'updated_rows', 'rejected_rows']

mark_batch_failed — line 175
  P

In [79]:
from pathlib import Path

# 1. Inspect the existing Spark transformations.
spark_file = SRC_DIR / "sales_pipeline" / "spark_transformations.py"

print("=" * 70)
print("EXISTING PYSPARK TRANSFORMATIONS")
print("=" * 70)
print(spark_file.read_text(encoding="utf-8"))

# 2. Identify existing storage-related modules.
package_dir = SRC_DIR / "sales_pipeline"

print("\n" + "=" * 70)
print("STORAGE-RELATED PROJECT MODULES")
print("=" * 70)

for file_path in sorted(package_dir.glob("*.py")):
    name = file_path.name.lower()
    if any(term in name for term in ["storage", "lake", "bronze", "silver", "gold"]):
        print(file_path.name)

# 3. Inspect the existing Phase 4 storage test structure.
test_file = TESTS_DIR / "test_phase4_storage_modeling.py"

if test_file.exists():
    print("\n" + "=" * 70)
    print("PHASE 4 STORAGE TESTS")
    print("=" * 70)
    print(test_file.read_text(encoding="utf-8"))

EXISTING PYSPARK TRANSFORMATIONS
"""Reusable PySpark transformations for sales analytics."""

from pyspark.sql import DataFrame
from pyspark.sql import functions as F


EXCLUDED_SALES_STATUSES = ("canceled", "unavailable")


def build_order_analytics_spark(
    orders: DataFrame,
    order_items: DataFrame,
    payments: DataFrame,
    customers: DataFrame,
) -> DataFrame:
    """Build one row per order with item, payment, customer, and delivery metrics."""

    item_metrics = order_items.groupBy("order_id").agg(
        F.count("order_item_id").alias("item_count"),
        F.countDistinct("product_id").alias("unique_products"),
        F.countDistinct("seller_id").alias("unique_sellers"),
        F.sum("price").alias("item_revenue"),
        F.sum("freight_value").alias("freight_cost"),
    )

    payment_metrics = payments.groupBy("order_id").agg(
        F.sum("payment_value").alias("payment_total"),
        F.count("payment_sequential").alias("payment_records"),
        F.countDist

In [80]:
from pathlib import Path

layers = ["bronze", "silver", "gold"]

for layer in layers:
    layer_path = DATA_DIR / layer

    print("\n" + "=" * 65)
    print(f"{layer.upper()} DATA LAYER")
    print("=" * 65)

    if not layer_path.exists():
        print("Directory not found:", layer_path)
        continue

    for path in sorted(layer_path.rglob("*")):
        relative = path.relative_to(DATA_DIR)

        if path.is_dir():
            parquet_count = sum(
                1 for item in path.glob("*.parquet")
                if item.is_file()
            )
            print(f"DIR  {relative} | Parquet files: {parquet_count}")

        elif path.is_file() and path.suffix == ".parquet":
            print(f"FILE {relative} | {path.stat().st_size:,} bytes")


BRONZE DATA LAYER
DIR  bronze/olist | Parquet files: 0
DIR  bronze/olist/olist_customers_dataset | Parquet files: 2
FILE bronze/olist/olist_customers_dataset/part-00000-6fb03181-5708-4362-bacf-c830c878fc80-c000.snappy.parquet | 5,122,037 bytes
FILE bronze/olist/olist_customers_dataset/part-00001-6fb03181-5708-4362-bacf-c830c878fc80-c000.snappy.parquet | 1,907,215 bytes
DIR  bronze/olist/olist_geolocation_dataset | Parquet files: 2
FILE bronze/olist/olist_geolocation_dataset/part-00000-070cfc71-c132-4737-9309-be6c24464ebc-c000.snappy.parquet | 0 bytes
FILE bronze/olist/olist_geolocation_dataset/part-00001-070cfc71-c132-4737-9309-be6c24464ebc-c000.snappy.parquet | 0 bytes
DIR  bronze/olist/olist_order_items_dataset | Parquet files: 2
FILE bronze/olist/olist_order_items_dataset/part-00000-1e3155a0-ffdf-45ff-ace4-3b0e80ff5f37-c000.snappy.parquet | 4,082,428 bytes
FILE bronze/olist/olist_order_items_dataset/part-00001-1e3155a0-ffdf-45ff-ace4-3b0e80ff5f37-c000.snappy.parquet | 2,529,619 byt

In [82]:
#Inspect R2 source objects
from sales_pipeline import r2_client

# Connect using existing Colab Secrets.
r2 = r2_client.create_r2_client_from_colab()
bucket = r2_client.get_bucket_name_from_colab()

source_prefixes = [
    "silver/orders/",
    "silver/order_items/",
    "silver/payments/",
    "silver/customers/",
    "silver/products/",
    "silver/category_translation/",
]

print("R2 SOURCE OBJECT INVENTORY")
print("=" * 75)

inventory = {}

for prefix in source_prefixes:
    objects = list(
        r2_client.list_objects(
            client=r2,
            bucket_name=bucket,
            prefix=prefix,
        )
    )

    inventory[prefix] = objects

    total_bytes = sum(obj["Size"] for obj in objects)
    print(f"\n{prefix}")
    print(f"  Objects: {len(objects)}")
    print(f"  Total size: {total_bytes:,} bytes")

    for obj in objects:
        print(f"  {obj['Key']} | {obj['Size']:,} bytes")

print("\nR2 INVENTORY COMPLETED")

R2 SOURCE OBJECT INVENTORY

silver/orders/
  Objects: 2
  Total size: 10,789,833 bytes
  silver/orders/part-00000-a7e7ce83-3a7b-4a22-80ff-443ac2b0adc1-c000.snappy.parquet | 6,665,407 bytes
  silver/orders/part-00001-a7e7ce83-3a7b-4a22-80ff-443ac2b0adc1-c000.snappy.parquet | 4,124,426 bytes

silver/order_items/
  Objects: 2
  Total size: 6,612,047 bytes
  silver/order_items/part-00000-0895601a-6fcf-4f21-b20b-17f037b45800-c000.snappy.parquet | 4,082,428 bytes
  silver/order_items/part-00001-0895601a-6fcf-4f21-b20b-17f037b45800-c000.snappy.parquet | 2,529,619 bytes

silver/payments/
  Objects: 2
  Total size: 3,865,725 bytes
  silver/payments/part-00000-47eeea1f-ee72-4bdc-a4b6-412662a84edd-c000.snappy.parquet | 3,315,480 bytes
  silver/payments/part-00001-47eeea1f-ee72-4bdc-a4b6-412662a84edd-c000.snappy.parquet | 550,245 bytes

silver/customers/
  Objects: 2
  Total size: 7,029,252 bytes
  silver/customers/part-00000-814ad7c6-9000-4e6f-8787-854afcae29f3-c000.snappy.parquet | 5,122,037 byt

In [83]:
#Implement multi-file R2 reader
from pathlib import Path

r2_file = SRC_DIR / "sales_pipeline" / "r2_client.py"
source = r2_file.read_text(encoding="utf-8")

function_name = "read_parquet_prefix_from_r2"

if f"def {function_name}(" in source:
    print(f"{function_name} already exists. No changes made.")
else:
    new_function = '''

def read_parquet_prefix_from_r2(
    spark,
    client,
    bucket_name: str,
    prefix: str,
    local_dir: str | Path,
):
    """
    Download and verify all Parquet objects under an R2 prefix,
    then return them as one Spark DataFrame.

    Local files are retained so Spark can evaluate its lazy query plan.
    Only objects discovered under the requested R2 prefix are read.
    """
    prefix = prefix.strip("/")

    if not prefix:
        raise ValueError("A non-empty R2 prefix is required.")

    objects = [
        obj for obj in list_objects(
            client,
            bucket_name,
            prefix=prefix + "/",
        )
        if obj.get("Key", "").lower().endswith(".parquet")
    ]

    if not objects:
        raise FileNotFoundError(
            f"No Parquet objects found under R2 prefix: {prefix}"
        )

    staging_dir = (
        Path(local_dir)
        / "r2_cache"
        / prefix.replace("/", "__")
    )
    staging_dir.mkdir(parents=True, exist_ok=True)

    local_files = []

    for obj in objects:
        object_key = obj["Key"]
        relative_key = object_key[len(prefix):].lstrip("/")
        destination = staging_dir / relative_key

        download_file(
            client=client,
            bucket_name=bucket_name,
            object_key=object_key,
            destination=destination,
        )

        verification = verify_object(
            client=client,
            bucket_name=bucket_name,
            object_key=object_key,
            local_path=destination,
        )

        if not (
            verification["size_matches"]
            and verification["checksum_matches"]
        ):
            raise RuntimeError(
                f"R2 download verification failed: {object_key}"
            )

        local_files.append(str(destination))

    return spark.read.parquet(*local_files)
'''

    with r2_file.open("a", encoding="utf-8") as f:
        f.write(new_function)

    print(f"Added {function_name}() successfully.")
    print("File:", r2_file)

Added read_parquet_prefix_from_r2() successfully.
File: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/r2_client.py


In [87]:
#reload the updated module

import importlib
from sales_pipeline import r2_client

# Reload the module from the updated source file.
r2_client = importlib.reload(r2_client)

print("Module reloaded:", r2_client.__file__)

if hasattr(r2_client, "read_parquet_prefix_from_r2"):
    print("PASS: read_parquet_prefix_from_r2() is available.")
else:
    raise AttributeError(
        "Function not found after reload. Check the source file."
    )

Module reloaded: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/r2_client.py
PASS: read_parquet_prefix_from_r2() is available.


In [88]:
#test all six Silver datasets
import sys
from pathlib import Path
from pyspark.sql import SparkSession
from sales_pipeline import r2_client

sys.path.insert(0, str(SRC_DIR))

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Phase7_R2_MultiFile_Read_Test")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

client = r2_client.create_r2_client_from_colab()
bucket = r2_client.get_bucket_name_from_colab()

prefixes = {
    "orders": "silver/orders",
    "order_items": "silver/order_items",
    "payments": "silver/payments",
    "customers": "silver/customers",
    "products": "silver/products",
    "category_translation": "silver/category_translation",
}

test_cache = PROJECT_ROOT / "data" / "phase7_test_cache"
results = {}

for name, prefix in prefixes.items():
    print(f"\nReading {name}...")

    df = r2_client.read_parquet_prefix_from_r2(
        spark=spark,
        client=client,
        bucket_name=bucket,
        prefix=prefix,
        local_dir=test_cache,
    )

    row_count = df.count()
    column_count = len(df.columns)

    results[name] = {
        "rows": row_count,
        "columns": column_count,
    }

    print(f"Rows: {row_count:,}")
    print(f"Columns: {column_count}")
    print("Schema:", df.columns)

print("\n" + "=" * 60)
print("PHASE 7.4 — MULTI-FILE R2 READ TEST")
print("=" * 60)

for name, result in results.items():
    print(
        f"{name:22} "
        f"{result['rows']:>10,} rows | "
        f"{result['columns']:>3} columns"
    )

assert all(result["rows"] > 0 for result in results.values())
print("\nALL SIX DATASETS READ SUCCESSFULLY.")


Reading orders...
Rows: 99,441
Columns: 8
Schema: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Reading order_items...
Rows: 112,650
Columns: 7
Schema: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Reading payments...
Rows: 103,886
Columns: 5
Schema: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Reading customers...
Rows: 99,441
Columns: 5
Schema: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Reading products...
Rows: 32,951
Columns: 9
Schema: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Reading category_translation...
Rows: 71
Co

In [89]:
#Execute existing PySpark transformations on R2 data
from sales_pipeline import r2_client
from sales_pipeline import spark_transformations as spark_tf

# Reload updated R2 module.
import importlib
r2_client = importlib.reload(r2_client)

# Read the six Silver datasets from R2.
prefixes = {
    "orders": "silver/orders",
    "order_items": "silver/order_items",
    "payments": "silver/payments",
    "customers": "silver/customers",
    "products": "silver/products",
    "category_translation": "silver/category_translation",
}

silver = {}

for name, prefix in prefixes.items():
    print(f"Loading {name}...")

    silver[name] = r2_client.read_parquet_prefix_from_r2(
        spark=spark,
        client=client,
        bucket_name=bucket,
        prefix=prefix,
        local_dir=test_cache,
    )

# Execute existing Spark transformations.
print("\nBuilding order analytics...")
order_gold = spark_tf.build_order_analytics_spark(
    orders=silver["orders"],
    order_items=silver["order_items"],
    payments=silver["payments"],
    customers=silver["customers"],
)

print("Building sales items...")
sales_items = spark_tf.prepare_sales_items_spark(
    orders=silver["orders"],
    order_items=silver["order_items"],
    products=silver["products"],
    category_translations=silver["category_translation"],
)

print("Building product analytics...")
product_gold = spark_tf.build_product_analytics_spark(sales_items)

print("Building category analytics...")
category_gold = spark_tf.build_category_analytics_spark(sales_items)

# Validate and report.
gold_outputs = {
    "order_analytics": order_gold,
    "product_analytics": product_gold,
    "category_analytics": category_gold,
}

print("\n" + "=" * 65)
print("PHASE 7.4 — SPARK TRANSFORMATION RESULTS")
print("=" * 65)

for name, df in gold_outputs.items():
    count = df.count()

    print(f"\n{name}")
    print("Rows:", f"{count:,}")
    print("Columns:", len(df.columns))
    print("Schema:", df.columns)

    assert count > 0, f"{name} is empty."

# Basic business-key validation.
assert order_gold.select("order_id").distinct().count() == order_gold.count()

print("\nALL THREE GOLD TRANSFORMATIONS EXECUTED SUCCESSFULLY.")

Loading orders...
Loading order_items...
Loading payments...
Loading customers...
Loading products...
Loading category_translation...

Building order analytics...
Building sales items...
Building product analytics...
Building category analytics...

PHASE 7.4 — SPARK TRANSFORMATION RESULTS

order_analytics
Rows: 99,441
Columns: 24
Schema: ['customer_id', 'order_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'item_count', 'unique_products', 'unique_sellers', 'item_revenue', 'freight_cost', 'payment_total', 'payment_records', 'payment_types', 'customer_unique_id', 'customer_city', 'customer_state', 'delivery_duration_days', 'estimated_delivery_duration_days', 'delivery_delay_days', 'is_delivered', 'is_late_delivery']

product_analytics
Rows: 32,729
Columns: 7
Schema: ['product_id', 'product_category_name_english', 'units_sold', 'item_revenue', 'freight_cost', 'distinct_

In [90]:
# reconcile against existing Gold data
from pyspark.sql import functions as F

local_gold = DATA_DIR / "gold" / "analytics"

comparisons = {
    "order_analytics": {
        "new": order_gold,
        "key": "order_id",
        "metrics": [
            "item_count",
            "unique_products",
            "unique_sellers",
            "item_revenue",
            "freight_cost",
            "delivery_duration_days",
            "estimated_delivery_duration_days",
            "delivery_delay_days",
        ],
    },
    "product_analytics": {
        "new": product_gold,
        "key": "product_id",
        "metrics": [
            "units_sold",
            "item_revenue",
            "freight_cost",
            "distinct_orders",
            "distinct_sellers",
        ],
    },
    "category_analytics": {
        "new": category_gold,
        "key": "product_category_name_english",
        "metrics": [
            "units_sold",
            "item_revenue",
            "freight_cost",
            "distinct_orders",
            "distinct_products",
            "distinct_sellers",
        ],
    },
}

reconciliation_results = {}

for name, config in comparisons.items():
    print(f"\nReconciling {name}...")

    new_df = config["new"]
    old_df = spark.read.parquet(str(local_gold / name))

    key = config["key"]
    metrics = [
        column for column in config["metrics"]
        if column in new_df.columns and column in old_df.columns
    ]

    # Compare row counts and business-key coverage.
    new_count = new_df.count()
    old_count = old_df.count()

    duplicate_new = (
        new_df.groupBy(key).count()
        .filter(F.col("count") > 1).count()
    )
    duplicate_old = (
        old_df.groupBy(key).count()
        .filter(F.col("count") > 1).count()
    )

    new_keys = new_df.select(key).distinct()
    old_keys = old_df.select(key).distinct()

    missing_from_new = old_keys.join(
        new_keys, on=key, how="left_anti"
    ).count()

    unexpected_in_new = new_keys.join(
        old_keys, on=key, how="left_anti"
    ).count()

    # Join by business key and compare metric values.
    joined = new_df.select(
        key,
        *[F.col(c).alias(f"new_{c}") for c in metrics]
    ).join(
        old_df.select(
            key,
            *[F.col(c).alias(f"old_{c}") for c in metrics]
        ),
        on=key,
        how="inner",
    )

    metric_checks = {}

    for column in metrics:
        new_value = F.col(f"new_{column}").cast("double")
        old_value = F.col(f"old_{column}").cast("double")

        stats = joined.agg(
            F.max(
                F.when(
                    new_value.isNull() != old_value.isNull(),
                    1
                ).otherwise(0)
            ).alias("null_mismatch"),
            F.max(
                F.abs(new_value - old_value)
            ).alias("max_difference"),
        ).first()

        null_mismatch = int(stats["null_mismatch"] or 0)
        max_difference = float(stats["max_difference"] or 0.0)

        passed = (
            null_mismatch == 0
            and max_difference <= 0.01
        )

        metric_checks[column] = passed

        print(
            f"  {column}: "
            f"{'PASS' if passed else 'FAIL'} "
            f"(max difference: {max_difference:.8f})"
        )

    passed = (
        new_count == old_count
        and duplicate_new == 0
        and duplicate_old == 0
        and missing_from_new == 0
        and unexpected_in_new == 0
        and all(metric_checks.values())
    )

    reconciliation_results[name] = passed

    print(f"  New rows: {new_count:,}")
    print(f"  Existing rows: {old_count:,}")
    print(f"  Missing keys: {missing_from_new}")
    print(f"  Unexpected keys: {unexpected_in_new}")
    print(f"  Duplicate keys (new/existing): {duplicate_new}/{duplicate_old}")
    print("  RESULT:", "PASS" if passed else "FAIL")

print("\n" + "=" * 65)
print("PHASE 7.4 — GOLD RECONCILIATION SUMMARY")
print("=" * 65)

for name, passed in reconciliation_results.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

assert all(reconciliation_results.values()), (
    "Gold reconciliation failed. Do not upload outputs yet."
)

print("\nALL GOLD OUTPUTS RECONCILED SUCCESSFULLY.")


Reconciling order_analytics...
  item_count: PASS (max difference: 0.00000000)
  unique_products: PASS (max difference: 0.00000000)
  unique_sellers: PASS (max difference: 0.00000000)
  item_revenue: PASS (max difference: 0.00000000)
  freight_cost: PASS (max difference: 0.00000000)
  delivery_duration_days: PASS (max difference: 0.00000000)
  estimated_delivery_duration_days: PASS (max difference: 0.00000000)
  delivery_delay_days: PASS (max difference: 0.00000000)
  New rows: 99,441
  Existing rows: 99,441
  Missing keys: 0
  Unexpected keys: 0
  Duplicate keys (new/existing): 0/0
  RESULT: PASS

Reconciling product_analytics...
  units_sold: PASS (max difference: 0.00000000)
  item_revenue: PASS (max difference: 0.00000000)
  freight_cost: PASS (max difference: 0.00000000)
  distinct_orders: PASS (max difference: 0.00000000)
  distinct_sellers: PASS (max difference: 0.00000000)
  New rows: 32,729
  Existing rows: 32,729
  Missing keys: 0
  Unexpected keys: 0
  Duplicate keys (new/e

In [91]:
#Upload reconciled Gold outputs to R2
from pathlib import Path
from sales_pipeline.r2_client import write_spark_parquet_to_r2

STAGING_DIR = PROJECT_ROOT / "data" / "phase7_staging"
STAGING_DIR.mkdir(parents=True, exist_ok=True)

gold_outputs = {
    "order_analytics": (
        order_gold,
        "phase7/gold/analytics/order_analytics/"
    ),
    "product_analytics": (
        product_gold,
        "phase7/gold/analytics/product_analytics/"
    ),
    "category_analytics": (
        category_gold,
        "phase7/gold/analytics/category_analytics/"
    ),
}

upload_results = {}

for dataset_name, (dataframe, destination_prefix) in gold_outputs.items():
    print(f"\nUploading {dataset_name}...")

    result = write_spark_parquet_to_r2(
        dataframe=dataframe,
        client=client,
        bucket_name=bucket,
        prefix=destination_prefix,
        local_dir=STAGING_DIR,
        mode="error",
    )

    upload_results[dataset_name] = result

    print(f"Uploaded files: {result['uploaded_file_count']}")
    print(f"Total size: {result['total_size_bytes']:,} bytes")
    print(f"Destination: {result['destination_prefix']}")
    print(f"Verification: {result['verification']}")

print("\n" + "=" * 60)
print("PHASE 7.4 — GOLD UPLOAD SUMMARY")
print("=" * 60)

for name, result in upload_results.items():
    print(
        f"{name}: {result['verification']} | "
        f"{result['uploaded_file_count']} files"
    )

assert all(
    result["verification"] == "PASS"
    for result in upload_results.values()
)

print("\nALL THREE GOLD DATASETS UPLOADED AND VERIFIED.")


Uploading order_analytics...
Uploaded files: 2
Total size: 16,936,180 bytes
Destination: phase7/gold/analytics/order_analytics
Verification: PASS

Uploading product_analytics...
Uploaded files: 2
Total size: 1,396,822 bytes
Destination: phase7/gold/analytics/product_analytics
Verification: PASS

Uploading category_analytics...
Uploaded files: 1
Total size: 5,828 bytes
Destination: phase7/gold/analytics/category_analytics
Verification: PASS

PHASE 7.4 — GOLD UPLOAD SUMMARY
order_analytics: PASS | 2 files
product_analytics: PASS | 2 files
category_analytics: PASS | 1 files

ALL THREE GOLD DATASETS UPLOADED AND VERIFIED.


In [93]:
#Independent R2 read-back verification
from sales_pipeline.r2_client import read_parquet_prefix_from_r2

READBACK_DIR = PROJECT_ROOT / "data" / "phase7_readback"
READBACK_DIR.mkdir(parents=True, exist_ok=True)

readback_sources = {
    "order_analytics": (
        order_gold,
        "phase7/gold/analytics/order_analytics/",
    ),
    "product_analytics": (
        product_gold,
        "phase7/gold/analytics/product_analytics/",
    ),
    "category_analytics": (
        category_gold,
        "phase7/gold/analytics/category_analytics/",
    ),
}

readback_results = {}

for name, (original_df, prefix) in readback_sources.items():
    print(f"\nVerifying {name}...")

    r2_df = read_parquet_prefix_from_r2(
        spark=spark,
        client=client,
        bucket_name=bucket,
        prefix=prefix,
        local_dir=READBACK_DIR,
    )

    original_columns = original_df.columns
    r2_columns = r2_df.columns

    assert set(original_columns) == set(r2_columns), (
        f"{name}: Column mismatch\n"
        f"Original: {original_columns}\n"
        f"R2: {r2_columns}"
    )

    original_aligned = original_df.select(*original_columns)
    r2_aligned = r2_df.select(*original_columns)

    original_count = original_aligned.count()
    r2_count = r2_aligned.count()

    original_only = original_aligned.exceptAll(r2_aligned).count()
    r2_only = r2_aligned.exceptAll(original_aligned).count()

    passed = (
        original_count == r2_count
        and original_only == 0
        and r2_only == 0
    )

    readback_results[name] = {
        "original_rows": original_count,
        "r2_rows": r2_count,
        "original_only": original_only,
        "r2_only": r2_only,
        "status": "PASS" if passed else "FAIL",
    }

    print(f"Original rows: {original_count:,}")
    print(f"R2 rows: {r2_count:,}")
    print(f"Rows missing from R2: {original_only:,}")
    print(f"Unexpected rows in R2: {r2_only:,}")
    print(f"RESULT: {'PASS' if passed else 'FAIL'}")

print("\n" + "=" * 60)
print("PHASE 7.4 — R2 GOLD READ-BACK SUMMARY")
print("=" * 60)

for name, result in readback_results.items():
    print(
        f"{name}: {result['status']} | "
        f"{result['r2_rows']:,} rows"
    )

assert all(
    result["status"] == "PASS"
    for result in readback_results.values()
), "One or more R2 read-back checks failed."

print("\nALL THREE R2 GOLD DATASETS MATCH THEIR ORIGINAL DATAFRAMES.")


Verifying order_analytics...
Original rows: 99,441
R2 rows: 99,441
Rows missing from R2: 0
Unexpected rows in R2: 0
RESULT: PASS

Verifying product_analytics...
Original rows: 32,729
R2 rows: 32,729
Rows missing from R2: 0
Unexpected rows in R2: 0
RESULT: PASS

Verifying category_analytics...
Original rows: 72
R2 rows: 72
Rows missing from R2: 0
Unexpected rows in R2: 0
RESULT: PASS

PHASE 7.4 — R2 GOLD READ-BACK SUMMARY
order_analytics: PASS | 99,441 rows
product_analytics: PASS | 32,729 rows
category_analytics: PASS | 72 rows

ALL THREE R2 GOLD DATASETS MATCH THEIR ORIGINAL DATAFRAMES.


In [94]:
#Build cloud_pipeline.py
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

CLOUD_PIPELINE_FILE = (
    PROJECT_ROOT / "src" / "sales_pipeline" / "cloud_pipeline.py"
)

cloud_pipeline_code = '''"""
Cloud pipeline orchestration for the Sales Analytics Data Engineering Pipeline.

Reads Silver datasets from Cloudflare R2, generates selected Gold analytics
using PySpark, and writes the outputs to dedicated Phase 7 R2 prefixes.
"""

from pathlib import Path
from typing import Any

from sales_pipeline import r2_client
from sales_pipeline import spark_transformations as spark_tf


SILVER_DATASETS = (
    "orders",
    "order_items",
    "payments",
    "customers",
    "products",
    "category_translation",
)

GOLD_PREFIXES = {
    "order_analytics": "phase7/gold/analytics/order_analytics/",
    "product_analytics": "phase7/gold/analytics/product_analytics/",
    "category_analytics": "phase7/gold/analytics/category_analytics/",
}


def run_cloud_pipeline(
    spark: Any,
    local_dir: str | Path,
    client: Any | None = None,
    bucket_name: str | None = None,
    write_mode: str = "error",
) -> dict:
    """
    Execute the Silver-to-Gold cloud pipeline using R2 object storage.

    The Spark session and R2 client can be injected for testing.
    Existing Phase 4 Gold objects are never targeted by this function.
    """

    if spark is None:
        raise ValueError("A valid Spark session is required.")

    if write_mode not in ("error", "overwrite"):
        raise ValueError("write_mode must be 'error' or 'overwrite'.")

    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)

    if client is None:
        client = r2_client.create_r2_client_from_colab()

    if bucket_name is None:
        bucket_name = r2_client.get_bucket_name_from_colab()

    r2_client.verify_bucket(client, bucket_name)

    silver = {}

    for dataset_name in SILVER_DATASETS:
        prefix = f"silver/{dataset_name}/"

        silver[dataset_name] = (
            r2_client.read_parquet_prefix_from_r2(
                spark=spark,
                client=client,
                bucket_name=bucket_name,
                prefix=prefix,
                local_dir=local_dir,
            )
        )

    order_gold = spark_tf.build_order_analytics_spark(
        orders=silver["orders"],
        order_items=silver["order_items"],
        payments=silver["payments"],
        customers=silver["customers"],
    )

    sales_items = spark_tf.prepare_sales_items_spark(
        orders=silver["orders"],
        order_items=silver["order_items"],
        products=silver["products"],
        category_translations=silver["category_translation"],
    )

    product_gold = spark_tf.build_product_analytics_spark(sales_items)
    category_gold = spark_tf.build_category_analytics_spark(sales_items)

    gold_outputs = {
        "order_analytics": order_gold,
        "product_analytics": product_gold,
        "category_analytics": category_gold,
    }

    upload_results = {}

    for dataset_name, dataframe in gold_outputs.items():
        upload_results[dataset_name] = (
            r2_client.write_spark_parquet_to_r2(
                dataframe=dataframe,
                client=client,
                bucket_name=bucket_name,
                prefix=GOLD_PREFIXES[dataset_name],
                local_dir=local_dir,
                mode=write_mode,
            )
        )

    return {
        "status": "SUCCESS",
        "bucket": bucket_name,
        "silver_datasets_read": list(SILVER_DATASETS),
        "gold_datasets_written": {
            name: {
                "row_count": dataframe.count(),
                "columns": dataframe.columns,
                "destination_prefix": GOLD_PREFIXES[name],
                "uploaded_file_count": upload_results[name][
                    "uploaded_file_count"
                ],
                "total_size_bytes": upload_results[name][
                    "total_size_bytes"
                ],
                "verification": upload_results[name]["verification"],
            }
            for name, dataframe in gold_outputs.items()
        },
    }
'''

CLOUD_PIPELINE_FILE.write_text(
    cloud_pipeline_code,
    encoding="utf-8"
)

print("Created:", CLOUD_PIPELINE_FILE)
print("File size:", CLOUD_PIPELINE_FILE.stat().st_size, "bytes")

Created: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/src/sales_pipeline/cloud_pipeline.py
File size: 3947 bytes


In [95]:
#Compile and validate the module.
import py_compile
import inspect
import sys

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 1. Compile the new module
py_compile.compile(
    str(CLOUD_PIPELINE_FILE),
    doraise=True
)
print("1. Python compilation: PASS")

# 2. Import the module
from sales_pipeline import cloud_pipeline

print("2. Module import: PASS")

# 3. Validate expected orchestration function
assert hasattr(cloud_pipeline, "run_cloud_pipeline")
print("3. run_cloud_pipeline exists: PASS")

print("\nFunction signature:")
print(inspect.signature(cloud_pipeline.run_cloud_pipeline))

# 4. Validate expected dataset configuration
assert len(cloud_pipeline.SILVER_DATASETS) == 6
assert len(cloud_pipeline.GOLD_PREFIXES) == 3

print("\nSilver datasets:", len(cloud_pipeline.SILVER_DATASETS))
print("Gold destinations:", len(cloud_pipeline.GOLD_PREFIXES))

print("\nMODULE VALIDATION COMPLETE.")

1. Python compilation: PASS
2. Module import: PASS
3. run_cloud_pipeline exists: PASS

Function signature:
(spark: Any, local_dir: str | pathlib._local.Path, client: typing.Any | None = None, bucket_name: str | None = None, write_mode: str = 'error') -> dict

Silver datasets: 6
Gold destinations: 3

MODULE VALIDATION COMPLETE.


In [96]:
#Add automated orchestration tests
import unittest
from unittest.mock import Mock, patch, call
from pathlib import Path

TEST_FILE = PROJECT_ROOT / "tests" / "test_cloud_pipeline.py"

test_code = '''import tempfile
import unittest
from unittest.mock import Mock, patch

from sales_pipeline import cloud_pipeline


class FakeDataFrame:
    def __init__(self, rows, columns):
        self.rows = rows
        self.columns = columns

    def count(self):
        return self.rows


class TestCloudPipeline(unittest.TestCase):

    def setUp(self):
        self.spark = Mock(name="spark")
        self.client = Mock(name="r2_client")
        self.bucket = "sales-analytics-pipeline"
        self.local_dir = tempfile.mkdtemp()

    def test_successful_cloud_pipeline(self):
        silver_frames = {
            name: FakeDataFrame(100, ["id"])
            for name in cloud_pipeline.SILVER_DATASETS
        }

        order_gold = FakeDataFrame(99441, ["order_id", "item_revenue"])
        product_gold = FakeDataFrame(32729, ["product_id", "item_revenue"])
        category_gold = FakeDataFrame(
            72, ["product_category_name_english", "item_revenue"]
        )
        sales_items = FakeDataFrame(112650, ["order_id", "product_id"])

        upload_result = {
            "uploaded_file_count": 2,
            "total_size_bytes": 12345,
            "verification": "PASS",
        }

        with (
            patch.object(
                cloud_pipeline.r2_client,
                "verify_bucket"
            ) as verify_bucket,
            patch.object(
                cloud_pipeline.r2_client,
                "read_parquet_prefix_from_r2",
                side_effect=lambda **kwargs: silver_frames[
                    kwargs["prefix"].split("/")[1]
                ],
            ) as read_silver,
            patch.object(
                cloud_pipeline.spark_tf,
                "build_order_analytics_spark",
                return_value=order_gold,
            ) as build_orders,
            patch.object(
                cloud_pipeline.spark_tf,
                "prepare_sales_items_spark",
                return_value=sales_items,
            ) as prepare_sales,
            patch.object(
                cloud_pipeline.spark_tf,
                "build_product_analytics_spark",
                return_value=product_gold,
            ) as build_products,
            patch.object(
                cloud_pipeline.spark_tf,
                "build_category_analytics_spark",
                return_value=category_gold,
            ) as build_categories,
            patch.object(
                cloud_pipeline.r2_client,
                "write_spark_parquet_to_r2",
                return_value=upload_result,
            ) as write_gold,
        ):
            result = cloud_pipeline.run_cloud_pipeline(
                spark=self.spark,
                local_dir=self.local_dir,
                client=self.client,
                bucket_name=self.bucket,
                write_mode="error",
            )

        verify_bucket.assert_called_once_with(self.client, self.bucket)
        self.assertEqual(read_silver.call_count, 6)

        build_orders.assert_called_once()
        prepare_sales.assert_called_once()
        build_products.assert_called_once_with(sales_items)
        build_categories.assert_called_once_with(sales_items)

        self.assertEqual(write_gold.call_count, 3)
        self.assertEqual(result["status"], "SUCCESS")
        self.assertEqual(result["bucket"], self.bucket)
        self.assertEqual(len(result["silver_datasets_read"]), 6)
        self.assertEqual(len(result["gold_datasets_written"]), 3)

        self.assertEqual(
            result["gold_datasets_written"]["order_analytics"]["row_count"],
            99441,
        )
        self.assertEqual(
            result["gold_datasets_written"]["product_analytics"]["row_count"],
            32729,
        )
        self.assertEqual(
            result["gold_datasets_written"]["category_analytics"]["row_count"],
            72,
        )

        self.assertEqual(
            result["gold_datasets_written"]["order_analytics"][
                "verification"
            ],
            "PASS",
        )

    def test_missing_spark_session_rejected(self):
        with self.assertRaises(ValueError):
            cloud_pipeline.run_cloud_pipeline(
                spark=None,
                local_dir=self.local_dir,
                client=self.client,
                bucket_name=self.bucket,
            )

    def test_invalid_write_mode_rejected(self):
        with self.assertRaises(ValueError):
            cloud_pipeline.run_cloud_pipeline(
                spark=self.spark,
                local_dir=self.local_dir,
                client=self.client,
                bucket_name=self.bucket,
                write_mode="invalid",
            )


if __name__ == "__main__":
    unittest.main()
'''

TEST_FILE.write_text(test_code, encoding="utf-8")
print("Created:", TEST_FILE)

# Run the new test module explicitly.
import sys
sys.path.insert(0, str(PROJECT_ROOT / "tests"))

import test_cloud_pipeline

suite = unittest.defaultTestLoader.loadTestsFromModule(
    test_cloud_pipeline
)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)

assert test_result.wasSuccessful(), "Cloud pipeline tests failed."

print("\\nCLOUD PIPELINE UNIT TESTS: ALL PASSED")

test_invalid_write_mode_rejected (test_cloud_pipeline.TestCloudPipeline.test_invalid_write_mode_rejected) ... ok
test_missing_spark_session_rejected (test_cloud_pipeline.TestCloudPipeline.test_missing_spark_session_rejected) ... ok
test_successful_cloud_pipeline (test_cloud_pipeline.TestCloudPipeline.test_successful_cloud_pipeline) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.011s

OK


Created: /content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/tests/test_cloud_pipeline.py
\nCLOUD PIPELINE UNIT TESTS: ALL PASSED


In [97]:
#Prepare a safe real-R2 integration run
from pathlib import Path

CLOUD_PIPELINE_FILE = (
    PROJECT_ROOT / "src" / "sales_pipeline" / "cloud_pipeline.py"
)

code = CLOUD_PIPELINE_FILE.read_text(encoding="utf-8")

old_signature = '''    bucket_name: str | None = None,
    write_mode: str = "error",
) -> dict:'''

new_signature = '''    bucket_name: str | None = None,
    write_mode: str = "error",
    gold_prefixes: dict | None = None,
) -> dict:'''

assert old_signature in code, "Expected function signature not found."

code = code.replace(old_signature, new_signature, 1)

old_validation = '''    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
'''

new_validation = '''    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)

    destinations = (
        GOLD_PREFIXES if gold_prefixes is None else gold_prefixes
    )

    if set(destinations) != set(GOLD_PREFIXES):
        raise ValueError(
            "gold_prefixes must contain exactly the configured Gold datasets."
        )

    if any(
        not isinstance(prefix, str) or not prefix.strip()
        for prefix in destinations.values()
    ):
        raise ValueError("Every Gold destination prefix must be non-empty.")
'''

assert old_validation in code, "Expected validation block not found."
code = code.replace(old_validation, new_validation, 1)

code = code.replace(
    'prefix=GOLD_PREFIXES[dataset_name],',
    'prefix=destinations[dataset_name],',
    1,
)

code = code.replace(
    '"destination_prefix": GOLD_PREFIXES[name],',
    '"destination_prefix": destinations[name],',
    1,
)

CLOUD_PIPELINE_FILE.write_text(code, encoding="utf-8")

print("Updated cloud_pipeline.py with configurable Gold destinations.")



Updated cloud_pipeline.py with configurable Gold destinations.


In [98]:
# Rerun unit tests
import importlib
import unittest

from sales_pipeline import cloud_pipeline

importlib.reload(cloud_pipeline)

import test_cloud_pipeline

suite = unittest.defaultTestLoader.loadTestsFromModule(
    test_cloud_pipeline
)

result = unittest.TextTestRunner(verbosity=2).run(suite)

assert result.wasSuccessful(), "Cloud pipeline tests failed."

print("\nALL CLOUD PIPELINE TESTS PASSED.")

test_invalid_write_mode_rejected (test_cloud_pipeline.TestCloudPipeline.test_invalid_write_mode_rejected) ... ok
test_missing_spark_session_rejected (test_cloud_pipeline.TestCloudPipeline.test_missing_spark_session_rejected) ... ok
test_successful_cloud_pipeline (test_cloud_pipeline.TestCloudPipeline.test_successful_cloud_pipeline) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.012s

OK



ALL CLOUD PIPELINE TESTS PASSED.


In [99]:
# Execute the real cloud pipeline
import importlib
from pathlib import Path
from sales_pipeline import cloud_pipeline

importlib.reload(cloud_pipeline)

TEST_PREFIXES = {
    "order_analytics": "phase7-test/orchestrator-run/order_analytics/",
    "product_analytics": "phase7-test/orchestrator-run/product_analytics/",
    "category_analytics": "phase7-test/orchestrator-run/category_analytics/",
}

print("Starting real R2 cloud pipeline...")
print("Bucket:", bucket)
print("Test destinations:")
for name, prefix in TEST_PREFIXES.items():
    print(f"  {name}: {prefix}")

cloud_result = cloud_pipeline.run_cloud_pipeline(
    spark=spark,
    local_dir=PROJECT_ROOT / "data" / "phase7_orchestrator_cache",
    client=client,
    bucket_name=bucket,
    write_mode="error",
    gold_prefixes=TEST_PREFIXES,
)

print("\n" + "=" * 65)
print("REAL CLOUD PIPELINE EXECUTION REPORT")
print("=" * 65)

print("Status:", cloud_result["status"])
print("Bucket:", cloud_result["bucket"])
print("Silver datasets read:", len(cloud_result["silver_datasets_read"]))

for name, details in cloud_result["gold_datasets_written"].items():
    print(f"\n{name}:")
    print("  Rows:", f"{details['row_count']:,}")
    print("  Columns:", len(details["columns"]))
    print("  Uploaded files:", details["uploaded_file_count"])
    print("  Size:", f"{details['total_size_bytes']:,} bytes")
    print("  Destination:", details["destination_prefix"])
    print("  Verification:", details["verification"])

assert cloud_result["status"] == "SUCCESS"
assert len(cloud_result["silver_datasets_read"]) == 6
assert len(cloud_result["gold_datasets_written"]) == 3

print("\nREAL CLOUD PIPELINE EXECUTION: SUCCESS")

Starting real R2 cloud pipeline...
Bucket: sales-analytics-pipeline
Test destinations:
  order_analytics: phase7-test/orchestrator-run/order_analytics/
  product_analytics: phase7-test/orchestrator-run/product_analytics/
  category_analytics: phase7-test/orchestrator-run/category_analytics/

REAL CLOUD PIPELINE EXECUTION REPORT
Status: SUCCESS
Bucket: sales-analytics-pipeline
Silver datasets read: 6

order_analytics:
  Rows: 99,441
  Columns: 24
  Uploaded files: 2
  Size: 16,936,180 bytes
  Destination: phase7-test/orchestrator-run/order_analytics/
  Verification: PASS

product_analytics:
  Rows: 32,729
  Columns: 7
  Uploaded files: 2
  Size: 1,396,874 bytes
  Destination: phase7-test/orchestrator-run/product_analytics/
  Verification: PASS

category_analytics:
  Rows: 72
  Columns: 7
  Uploaded files: 1
  Size: 5,820 bytes
  Destination: phase7-test/orchestrator-run/category_analytics/
  Verification: PASS

REAL CLOUD PIPELINE EXECUTION: SUCCESS


In [100]:
#  Independently verify orchestrator outputs in R2
from sales_pipeline.r2_client import read_parquet_prefix_from_r2

ORCHESTRATOR_READBACK_DIR = (
    PROJECT_ROOT / "data" / "phase7_orchestrator_readback"
)
ORCHESTRATOR_READBACK_DIR.mkdir(parents=True, exist_ok=True)

expected_outputs = {
    "order_analytics": (
        order_gold,
        TEST_PREFIXES["order_analytics"],
    ),
    "product_analytics": (
        product_gold,
        TEST_PREFIXES["product_analytics"],
    ),
    "category_analytics": (
        category_gold,
        TEST_PREFIXES["category_analytics"],
    ),
}

verification_results = {}

for name, (expected_df, prefix) in expected_outputs.items():
    print(f"\nVerifying {name} from R2...")

    actual_df = read_parquet_prefix_from_r2(
        spark=spark,
        client=client,
        bucket_name=bucket,
        prefix=prefix,
        local_dir=ORCHESTRATOR_READBACK_DIR,
    )

    assert set(expected_df.columns) == set(actual_df.columns), (
        f"{name}: column mismatch"
    )

    expected_df = expected_df.select(*actual_df.columns)

    expected_rows = expected_df.count()
    actual_rows = actual_df.count()

    expected_only = expected_df.exceptAll(actual_df).count()
    actual_only = actual_df.exceptAll(expected_df).count()

    passed = (
        expected_rows == actual_rows
        and expected_only == 0
        and actual_only == 0
    )

    verification_results[name] = {
        "expected_rows": expected_rows,
        "actual_rows": actual_rows,
        "expected_only": expected_only,
        "actual_only": actual_only,
        "status": "PASS" if passed else "FAIL",
    }

    print(f"Expected rows: {expected_rows:,}")
    print(f"R2 rows: {actual_rows:,}")
    print(f"Missing rows: {expected_only:,}")
    print(f"Unexpected rows: {actual_only:,}")
    print(f"RESULT: {'PASS' if passed else 'FAIL'}")

print("\n" + "=" * 60)
print("ORCHESTRATOR READ-BACK VERIFICATION")
print("=" * 60)

for name, result in verification_results.items():
    print(
        f"{name}: {result['status']} | "
        f"{result['actual_rows']:,} rows"
    )

assert all(
    result["status"] == "PASS"
    for result in verification_results.values()
), "Orchestrator read-back verification failed."

print("\nALL ORCHESTRATOR OUTPUTS VERIFIED SUCCESSFULLY.")


Verifying order_analytics from R2...
Expected rows: 99,441
R2 rows: 99,441
Missing rows: 0
Unexpected rows: 0
RESULT: PASS

Verifying product_analytics from R2...
Expected rows: 32,729
R2 rows: 32,729
Missing rows: 0
Unexpected rows: 0
RESULT: PASS

Verifying category_analytics from R2...
Expected rows: 72
R2 rows: 72
Missing rows: 0
Unexpected rows: 0
RESULT: PASS

ORCHESTRATOR READ-BACK VERIFICATION
order_analytics: PASS | 99,441 rows
product_analytics: PASS | 32,729 rows
category_analytics: PASS | 72 rows

ALL ORCHESTRATOR OUTPUTS VERIFIED SUCCESSFULLY.


In [106]:
import unittest
import tempfile
import importlib
from contextlib import ExitStack
from unittest.mock import Mock, patch

TEST_FILE = PROJECT_ROOT / "tests" / "test_cloud_pipeline.py"

failure_tests = '''

class TestCloudPipelineFailureHandling(unittest.TestCase):

    def setUp(self):
        self.spark = Mock(name="spark")
        self.client = Mock(name="r2_client")
        self.bucket = "sales-analytics-pipeline"
        self.local_dir = tempfile.mkdtemp()

    def run_with_mocks(self, failure_stage=None):
        silver_frames = {
            name: FakeDataFrame(100, ["id"])
            for name in cloud_pipeline.SILVER_DATASETS
        }

        order_gold = FakeDataFrame(99441, ["order_id", "item_revenue"])
        product_gold = FakeDataFrame(32729, ["product_id", "item_revenue"])
        category_gold = FakeDataFrame(
            72, ["product_category_name_english", "item_revenue"]
        )
        sales_items = FakeDataFrame(112650, ["order_id", "product_id"])

        upload_result = {
            "uploaded_file_count": 2,
            "total_size_bytes": 12345,
            "verification": "PASS",
        }

        with ExitStack() as stack:
            verify_bucket = stack.enter_context(
                patch.object(cloud_pipeline.r2_client, "verify_bucket")
            )

            read_silver = stack.enter_context(
                patch.object(
                    cloud_pipeline.r2_client,
                    "read_parquet_prefix_from_r2",
                    side_effect=lambda **kwargs: silver_frames[
                        kwargs["prefix"].split("/")[1]
                    ],
                )
            )

            stack.enter_context(
                patch.object(
                    cloud_pipeline.spark_tf,
                    "build_order_analytics_spark",
                    return_value=order_gold,
                )
            )

            stack.enter_context(
                patch.object(
                    cloud_pipeline.spark_tf,
                    "prepare_sales_items_spark",
                    return_value=sales_items,
                )
            )

            stack.enter_context(
                patch.object(
                    cloud_pipeline.spark_tf,
                    "build_product_analytics_spark",
                    return_value=product_gold,
                )
            )

            stack.enter_context(
                patch.object(
                    cloud_pipeline.spark_tf,
                    "build_category_analytics_spark",
                    return_value=category_gold,
                )
            )

            write_gold = stack.enter_context(
                patch.object(
                    cloud_pipeline.r2_client,
                    "write_spark_parquet_to_r2",
                    return_value=upload_result,
                )
            )

            if failure_stage == "bucket":
                verify_bucket.side_effect = RuntimeError(
                    "Simulated R2 bucket verification failure"
                )

            elif failure_stage == "silver":
                read_silver.side_effect = FileNotFoundError(
                    "Simulated missing Silver dataset"
                )

            elif failure_stage == "upload":
                write_gold.side_effect = RuntimeError(
                    "Simulated R2 upload failure"
                )

            elif failure_stage == "existing_prefix":
                write_gold.side_effect = FileExistsError(
                    "Destination prefix already contains objects"
                )

            return cloud_pipeline.run_cloud_pipeline(
                spark=self.spark,
                local_dir=self.local_dir,
                client=self.client,
                bucket_name=self.bucket,
                write_mode="error",
            )

    def test_bucket_verification_failure(self):
        with self.assertRaisesRegex(
            RuntimeError, "bucket verification failure"
        ):
            self.run_with_mocks("bucket")

    def test_missing_silver_dataset(self):
        with self.assertRaisesRegex(
            FileNotFoundError, "missing Silver dataset"
        ):
            self.run_with_mocks("silver")

    def test_gold_upload_failure(self):
        with self.assertRaisesRegex(
            RuntimeError, "upload failure"
        ):
            self.run_with_mocks("upload")

    def test_existing_gold_prefix_rejected(self):
        with self.assertRaisesRegex(
            FileExistsError, "already contains objects"
        ):
            self.run_with_mocks("existing_prefix")

    def test_invalid_gold_prefix_configuration(self):
        invalid_prefixes = {
            "order_analytics": "phase7-test/orders/",
            "product_analytics": "phase7-test/products/",
        }

        with self.assertRaisesRegex(
            ValueError, "exactly the configured Gold datasets"
        ):
            cloud_pipeline.run_cloud_pipeline(
                spark=self.spark,
                local_dir=self.local_dir,
                client=self.client,
                bucket_name=self.bucket,
                gold_prefixes=invalid_prefixes,
            )
'''

existing_code = TEST_FILE.read_text(encoding="utf-8")

if "class TestCloudPipelineFailureHandling" not in existing_code:
    with TEST_FILE.open("a", encoding="utf-8") as file:
        file.write(failure_tests)

    print("Added five failure-handling tests.")
else:
    print("Failure-handling tests already exist; no duplicate added.")

import test_cloud_pipeline
importlib.reload(test_cloud_pipeline)

suite = unittest.defaultTestLoader.loadTestsFromModule(
    test_cloud_pipeline
)

result = unittest.TextTestRunner(verbosity=2).run(suite)

print("\nTotal tests:", result.testsRun)
print("Failures:", len(result.failures))
print("Errors:", len(result.errors))

assert result.wasSuccessful(), "One or more cloud pipeline tests failed."

print("\nALL CLOUD PIPELINE RELIABILITY TESTS PASSED.")

test_invalid_write_mode_rejected (test_cloud_pipeline.TestCloudPipeline.test_invalid_write_mode_rejected) ... ok
test_missing_spark_session_rejected (test_cloud_pipeline.TestCloudPipeline.test_missing_spark_session_rejected) ... ok
test_successful_cloud_pipeline (test_cloud_pipeline.TestCloudPipeline.test_successful_cloud_pipeline) ... ok
test_bucket_verification_failure (test_cloud_pipeline.TestCloudPipelineFailureHandling.test_bucket_verification_failure) ... ERROR
test_existing_gold_prefix_rejected (test_cloud_pipeline.TestCloudPipelineFailureHandling.test_existing_gold_prefix_rejected) ... ERROR
test_gold_upload_failure (test_cloud_pipeline.TestCloudPipelineFailureHandling.test_gold_upload_failure) ... ERROR
test_invalid_gold_prefix_configuration (test_cloud_pipeline.TestCloudPipelineFailureHandling.test_invalid_gold_prefix_configuration) ... ok
test_missing_silver_dataset (test_cloud_pipeline.TestCloudPipelineFailureHandling.test_missing_silver_dataset) ... ERROR

ERROR: test_buck

Failure-handling tests already exist; no duplicate added.

Total tests: 8
Failures: 0
Errors: 4


AssertionError: One or more cloud pipeline tests failed.

In [107]:
import importlib
import unittest

TEST_FILE = PROJECT_ROOT / "tests" / "test_cloud_pipeline.py"

test_source = TEST_FILE.read_text(encoding="utf-8")

required_import = "from contextlib import ExitStack\n"

if required_import not in test_source:
    TEST_FILE.write_text(
        required_import + test_source,
        encoding="utf-8"
    )
    print("Added missing ExitStack import.")
else:
    print("ExitStack import already exists.")

# Reload the updated test module
import test_cloud_pipeline
importlib.reload(test_cloud_pipeline)

suite = unittest.defaultTestLoader.loadTestsFromModule(
    test_cloud_pipeline
)

result = unittest.TextTestRunner(verbosity=2).run(suite)

print("\nTotal tests:", result.testsRun)
print("Failures:", len(result.failures))
print("Errors:", len(result.errors))

assert result.wasSuccessful(), "Cloud pipeline tests failed."

print("\nALL 8 CLOUD PIPELINE TESTS PASSED.")

test_invalid_write_mode_rejected (test_cloud_pipeline.TestCloudPipeline.test_invalid_write_mode_rejected) ... ok
test_missing_spark_session_rejected (test_cloud_pipeline.TestCloudPipeline.test_missing_spark_session_rejected) ... ok
test_successful_cloud_pipeline (test_cloud_pipeline.TestCloudPipeline.test_successful_cloud_pipeline) ... ok
test_bucket_verification_failure (test_cloud_pipeline.TestCloudPipelineFailureHandling.test_bucket_verification_failure) ... ok
test_existing_gold_prefix_rejected (test_cloud_pipeline.TestCloudPipelineFailureHandling.test_existing_gold_prefix_rejected) ... ok
test_gold_upload_failure (test_cloud_pipeline.TestCloudPipelineFailureHandling.test_gold_upload_failure) ... ok
test_invalid_gold_prefix_configuration (test_cloud_pipeline.TestCloudPipelineFailureHandling.test_invalid_gold_prefix_configuration) ... ok
test_missing_silver_dataset (test_cloud_pipeline.TestCloudPipelineFailureHandling.test_missing_silver_dataset) ... ok

----------------------------

Added missing ExitStack import.

Total tests: 8
Failures: 0
Errors: 0

ALL 8 CLOUD PIPELINE TESTS PASSED.


In [109]:
# Install the missing dependency
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "psycopg[binary]"
])

import psycopg

print("psycopg version:", psycopg.__version__)
print("PostgreSQL dependency installation: PASS")

psycopg version: 3.3.9
PostgreSQL dependency installation: PASS


In [110]:
# Finalize Phase 7.4 documentation
import unittest
import sys

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

if str(PROJECT_ROOT / "tests") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "tests"))

loader = unittest.TestLoader()

suite = loader.discover(
    start_dir=str(PROJECT_ROOT / "tests"),
    pattern="test_*.py",
    top_level_dir=str(PROJECT_ROOT),
)

result = unittest.TextTestRunner(verbosity=2).run(suite)

print("\n" + "=" * 55)
print("PROJECT-WIDE TEST SUMMARY")
print("=" * 55)
print("Tests run:", result.testsRun)
print("Failures:", len(result.failures))
print("Errors:", len(result.errors))
print("Skipped:", len(result.skipped))

assert result.wasSuccessful(), "Project-wide regression tests failed."

print("\nPROJECT-WIDE REGRESSION TESTS PASSED.")


test_invalid_write_mode_rejected (tests.test_cloud_pipeline.TestCloudPipeline.test_invalid_write_mode_rejected) ... ok
test_missing_spark_session_rejected (tests.test_cloud_pipeline.TestCloudPipeline.test_missing_spark_session_rejected) ... ok
test_successful_cloud_pipeline (tests.test_cloud_pipeline.TestCloudPipeline.test_successful_cloud_pipeline) ... ok
test_bucket_verification_failure (tests.test_cloud_pipeline.TestCloudPipelineFailureHandling.test_bucket_verification_failure) ... ok
test_existing_gold_prefix_rejected (tests.test_cloud_pipeline.TestCloudPipelineFailureHandling.test_existing_gold_prefix_rejected) ... ok
test_gold_upload_failure (tests.test_cloud_pipeline.TestCloudPipelineFailureHandling.test_gold_upload_failure) ... ok
test_invalid_gold_prefix_configuration (tests.test_cloud_pipeline.TestCloudPipelineFailureHandling.test_invalid_gold_prefix_configuration) ... ok
test_missing_silver_dataset (tests.test_cloud_pipeline.TestCloudPipelineFailureHandling.test_missing_silv

SALES ANALYTICS DATA PIPELINE

[1/5] Validating source data...

[2/5] Building order analytics...

[3/5] Building product and category analytics...

[4/5] Building customer analytics...

[5/5] Building sales performance analytics...

PIPELINE COMPLETED SUCCESSFULLY
Duration: 0.01 seconds
Validation checks: 1
Order rows: 2
Product rows: 1
Category rows: 1
Customer rows: 1
Monthly records: 1
Order status records: 1

Generated outputs:
- /tmp/tmpqhvvabkn/processed/order_analytics.csv (5 bytes)
- /tmp/tmpqhvvabkn/processed/product_analytics.csv (5 bytes)
- /tmp/tmpqhvvabkn/processed/category_analytics.csv (5 bytes)
- /tmp/tmpqhvvabkn/processed/customer_analytics.csv (5 bytes)
- /tmp/tmpqhvvabkn/processed/monthly_sales_analytics.csv (5 bytes)
- /tmp/tmpqhvvabkn/processed/order_status_summary.csv (5 bytes)
SALES ANALYTICS DATA PIPELINE

[1/5] Validating source data...

[2/5] Building order analytics...

[3/5] Building product and category analytics...

[4/5] Building customer analytics...

[

test_category_names_are_unique (tests.test_pipeline.TestPipelineOutputs.test_category_names_are_unique) ... ok
test_customer_ids_are_unique (tests.test_pipeline.TestPipelineOutputs.test_customer_ids_are_unique) ... ok
test_customer_repeat_flag (tests.test_pipeline.TestPipelineOutputs.test_customer_repeat_flag) ... ok
test_expected_outputs_exist (tests.test_pipeline.TestPipelineOutputs.test_expected_outputs_exist) ... ok
test_monthly_order_month_is_unique (tests.test_pipeline.TestPipelineOutputs.test_monthly_order_month_is_unique) ... ok
test_order_ids_are_unique (tests.test_pipeline.TestPipelineOutputs.test_order_ids_are_unique) ... ok
test_order_status_counts_reconcile (tests.test_pipeline.TestPipelineOutputs.test_order_status_counts_reconcile) ... ok
test_product_and_category_revenue_reconcile (tests.test_pipeline.TestPipelineOutputs.test_product_and_category_revenue_reconcile) ... ok
test_product_and_category_units_reconcile (tests.test_pipeline.TestPipelineOutputs.test_product_and_


PROJECT-WIDE TEST SUMMARY
Tests run: 63
Failures: 0
Errors: 0
Skipped: 1

PROJECT-WIDE REGRESSION TESTS PASSED.


#### Production readiness and documentation

In [111]:
from pathlib import Path

DOCS_DIR = PROJECT_ROOT / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

report = """# Phase 7 — Cloud Deployment Extension

## 1. Architecture

| Component | Responsibility |
|---|---|
| Google Colab | Development and pipeline orchestration |
| Cloudflare R2 | Bronze, Silver and Gold Parquet storage |
| PySpark | Data transformation and analytics |
| Neon PostgreSQL | Existing analytical warehouse |
| GitHub | Source code and version control |

## 2. Data Flow

1. Source datasets are stored in the Bronze layer.
2. Silver and Gold Parquet datasets are stored in Cloudflare R2.
3. PySpark reads six Silver datasets from R2.
4. Transformations generate order, product and category analytics.
5. Gold outputs are uploaded to dedicated R2 prefixes.
6. Uploaded data is independently read back and reconciled.

## 3. Security and Access

- R2 access uses bucket-scoped Object Read and Write credentials.
- Credentials are stored in Google Colab Secrets.
- Secret values are never stored in source code or GitHub.
- Existing R2 access permissions are restricted to the project bucket.
- Neon PostgreSQL credentials remain separately managed.

## 4. Reliability and Validation

- SHA256 checksum verification for uploaded objects.
- Existing-object protection using error mode.
- Multi-file Parquet read and write support.
- Spark output reconciliation against existing Gold data.
- Independent R2 read-back verification.
- Automated failure-handling tests.

## 5. Implementation Evidence

- Six Silver datasets successfully read from R2.
- Three Gold datasets generated using PySpark.
- 132,242 Gold rows independently reconciled.
- Five Parquet objects uploaded in the orchestrator test.
- Eight cloud orchestration tests passed.
- Project-wide regression suite: 63 tests, zero failures and errors.
- One database integration test skipped because NEON_DB_URL was unavailable.

## 6. Limitations

- PySpark execution is currently performed in Google Colab, not a managed Spark cluster.
- Cloudflare R2 provides object storage, not managed ETL orchestration.
- The cloud orchestrator currently generates three Gold analytics datasets.
- Neon PostgreSQL remains the existing warehouse; the new cloud orchestrator does not load its outputs into Neon.
- Continuous scheduling and production monitoring are not implemented.

## 7. Cost Considerations

- No new paid compute service was introduced for this implementation.
- Colab and R2 usage remain subject to their respective account limits and billing policies.
- Monitor storage, operations, and any applicable account charges.
- No additional paid infrastructure is required by the current implementation.

## 8. Conclusion

Phase 7 successfully extends the Sales Analytics Data Engineering Pipeline
with Cloudflare R2 storage, PySpark integration, reusable cloud orchestration,
and independently verified Gold analytics outputs.
"""

REPORT_FILE = DOCS_DIR / "phase7_cloud_deployment.md"
REPORT_FILE.write_text(report, encoding="utf-8")

print("Phase 7 documentation created:")
print(REPORT_FILE)
print("Size:", REPORT_FILE.stat().st_size, "bytes")

Phase 7 documentation created:
/content/drive/MyDrive/Sales-analytics-data-pipeline/Sales-analytics-data-pipeline/docs/phase7_cloud_deployment.md
Size: 2797 bytes


#### Github Push

In [112]:
# Check git status
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        check=True
    ).stdout.strip()

print("BRANCH:", git("branch", "--show-current"))
print("\nREMOTE:")
print(git("remote", "-v"))

print("\nCHANGED FILES:")
print(git("status", "--short"))

print("\nCHANGE SUMMARY:")
print(git("diff", "--stat"))

print("\nUNTRACKED FILES:")
print(git("ls-files", "--others", "--exclude-standard"))

BRANCH: main

REMOTE:
origin	https://github.com/vipulkumawat17/Sales-analytics-data-pipeline (fetch)
origin	https://github.com/vipulkumawat17/Sales-analytics-data-pipeline (push)

CHANGED FILES:
M 06_Analytics_and_Performance_Evaluation.ipynb
?? 07_Cloudflare_R2_Deployment.ipynb
?? docs/phase7_cloud_deployment.md
?? src/sales_pipeline/cloud_pipeline.py
?? src/sales_pipeline/r2_client.py
?? tests/test_cloud_pipeline.py
?? tests/test_r2_client.py

CHANGE SUMMARY:
06_Analytics_and_Performance_Evaluation.ipynb | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)

UNTRACKED FILES:
07_Cloudflare_R2_Deployment.ipynb
docs/phase7_cloud_deployment.md
src/sales_pipeline/cloud_pipeline.py
src/sales_pipeline/r2_client.py
tests/test_cloud_pipeline.py
tests/test_r2_client.py


In [113]:
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

def run_git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )

# 1. Check whitespace and patch errors
check = run_git("diff", "--check")
print("Git diff check:", "PASS" if check.returncode == 0 else "REVIEW")
print(check.stdout or check.stderr)

# 2. Inspect the existing notebook modification
print("\nPHASE 6 NOTEBOOK DIFF:")
diff = run_git("diff", "--", "06_Analytics_and_Performance_Evaluation.ipynb")
print(diff.stdout[:5000] if diff.stdout else "No textual diff.")

# 3. Check candidate file sizes
files = [
    "07_Cloudflare_R2_Deployment.ipynb",
    "docs/phase7_cloud_deployment.md",
    "src/sales_pipeline/cloud_pipeline.py",
    "src/sales_pipeline/r2_client.py",
    "tests/test_cloud_pipeline.py",
    "tests/test_r2_client.py",
]

print("\nFILE SIZES:")
for name in files:
    path = PROJECT_ROOT / name
    if path.exists():
        print(f"{name}: {path.stat().st_size:,} bytes")

# 4. Check for common secret patterns without printing matched values
import re

patterns = {
    "AWS-style access key": re.compile(r"AKIA[0-9A-Z]{16}"),
    "Private key": re.compile(r"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----"),
    "Possible API token": re.compile(r"(?:api_key|access_token|secret_access_key)\s*[:=]\s*['\"][^'\"]{12,}"),
    "Database URL with credentials": re.compile(r"postgres(?:ql)?://[^/\s:]+:[^@\s]+@"),
}

print("\nSECRET SCAN:")
for name in files:
    path = PROJECT_ROOT / name
    if not path.exists():
        continue

    content = path.read_text(encoding="utf-8", errors="ignore")
    findings = [label for label, pattern in patterns.items()
                if pattern.search(content)]

    print(f"{name}: {'REVIEW REQUIRED - ' + ', '.join(findings) if findings else 'No common secret patterns found'}")

Git diff check: PASS


PHASE 6 NOTEBOOK DIFF:
diff --git a/06_Analytics_and_Performance_Evaluation.ipynb b/06_Analytics_and_Performance_Evaluation.ipynb
index ea3d727..24bc5ea 100644
--- a/06_Analytics_and_Performance_Evaluation.ipynb
+++ b/06_Analytics_and_Performance_Evaluation.ipynb
@@ -1 +1 @@
-{"nbformat":4,"nbformat_minor":0,"metadata":{"colab":{"provenance":[],"authorship_tag":"ABX9TyMHGyV3J8k4goqzf9t/U+2i"},"kernelspec":{"name":"python3","display_name":"Python 3"},"language_info":{"name":"python"}},"cells":[{"cell_type":"markdown","source":["## Inspect Project"],"metadata":{"id":"1XV0tr11U554"}},{"cell_type":"code","execution_count":4,"metadata":{"id":"TabqmFMYTXps","colab":{"base_uri":"https://localhost:8080/"},"executionInfo":{"status":"ok","timestamp":1793351938654,"user_tz":-330,"elapsed":28199,"user":{"displayName":"Vipul Kumawat","userId":"10244865491502480161"}},"outputId":"ad83d525-6e65-49fc-d886-f6c420140228"},"outputs":[{"output_type":"stream","name":"stdout","text":[

In [114]:
import subprocess

def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        check=True
    ).stdout.strip()

# Restore Phase 6 notebook to its original committed version
git(
    "restore",
    "--source=HEAD",
    "--staged",
    "--worktree",
    "--",
    "06_Analytics_and_Performance_Evaluation.ipynb"
)

# Stage only Phase 7 files
phase7_files = [
    "07_Cloudflare_R2_Deployment.ipynb",
    "docs/phase7_cloud_deployment.md",
    "src/sales_pipeline/cloud_pipeline.py",
    "src/sales_pipeline/r2_client.py",
    "tests/test_cloud_pipeline.py",
    "tests/test_r2_client.py",
]

git("add", "--", *phase7_files)

print("STAGED FILES:")
print(git("diff", "--cached", "--name-status"))

print("\nSTAGED CHANGE SUMMARY:")
print(git("diff", "--cached", "--stat"))

print("\nREMAINING GIT STATUS:")
print(git("status", "--short"))

STAGED FILES:
A	07_Cloudflare_R2_Deployment.ipynb
A	docs/phase7_cloud_deployment.md
A	src/sales_pipeline/cloud_pipeline.py
A	src/sales_pipeline/r2_client.py
A	tests/test_cloud_pipeline.py
A	tests/test_r2_client.py

STAGED CHANGE SUMMARY:
07_Cloudflare_R2_Deployment.ipynb    |   1 +
 docs/phase7_cloud_deployment.md      |  68 +++++
 src/sales_pipeline/cloud_pipeline.py | 149 +++++++++++
 src/sales_pipeline/r2_client.py      | 473 +++++++++++++++++++++++++++++++++++
 tests/test_cloud_pipeline.py         | 300 ++++++++++++++++++++++
 tests/test_r2_client.py              | 417 ++++++++++++++++++++++++++++++
 6 files changed, 1408 insertions(+)

REMAINING GIT STATUS:
A  07_Cloudflare_R2_Deployment.ipynb
A  docs/phase7_cloud_deployment.md
A  src/sales_pipeline/cloud_pipeline.py
A  src/sales_pipeline/r2_client.py
A  tests/test_cloud_pipeline.py
A  tests/test_r2_client.py


In [120]:
import subprocess

def git_check(*args):
    return subprocess.run(
        ["git", *args],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )

print("GIT STATUS:")
print(git_check("status", "--short").stdout)

print("\nUNSTAGED MODIFIED FILES:")
print(git_check("diff", "--name-only").stdout or "None")

print("\nUNSTAGED CHANGE SUMMARY:")
print(git_check("diff", "--stat").stdout or "None")

print("\nSTAGED FILES:")
print(git_check("diff", "--cached", "--name-only").stdout)

GIT STATUS:
AM 07_Cloudflare_R2_Deployment.ipynb
A  docs/phase7_cloud_deployment.md
A  src/sales_pipeline/cloud_pipeline.py
A  src/sales_pipeline/r2_client.py
A  tests/test_cloud_pipeline.py
A  tests/test_r2_client.py


UNSTAGED MODIFIED FILES:
07_Cloudflare_R2_Deployment.ipynb


UNSTAGED CHANGE SUMMARY:
 07_Cloudflare_R2_Deployment.ipynb | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


STAGED FILES:
07_Cloudflare_R2_Deployment.ipynb
docs/phase7_cloud_deployment.md
src/sales_pipeline/cloud_pipeline.py
src/sales_pipeline/r2_client.py
tests/test_cloud_pipeline.py
tests/test_r2_client.py



In [123]:
import subprocess

result = subprocess.run(
    ["git", "diff", "--numstat", "--", "07_Cloudflare_R2_Deployment.ipynb"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print("Notebook change summary:")
print(result.stdout or "No unstaged changes.")

Notebook change summary:
1	1	07_Cloudflare_R2_Deployment.ipynb



In [119]:
import os
import subprocess
import tempfile
from pathlib import Path
from google.colab import userdata

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Sales-analytics-data-pipeline/"
    "Sales-analytics-data-pipeline"
)

def git(*args, env=None):
    return subprocess.run(
        ["git", *args],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        check=True,
        env=env
    )

# 1. Load GitHub token securely
github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise ValueError("GITHUB_TOKEN is missing from Colab Secrets.")

print("GitHub token loaded securely.")

# 2. Verify staged files
expected_files = {
    "07_Cloudflare_R2_Deployment.ipynb",
    "docs/phase7_cloud_deployment.md",
    "src/sales_pipeline/cloud_pipeline.py",
    "src/sales_pipeline/r2_client.py",
    "tests/test_cloud_pipeline.py",
    "tests/test_r2_client.py",
}

staged_files = set(
    git("diff", "--cached", "--name-only").stdout.splitlines()
)

assert staged_files == expected_files, (
    f"Unexpected staged files: {staged_files ^ expected_files}"
)

# Ensure no unstaged tracked modifications
git("diff", "--quiet")

# 3. Verify Git identity
git_name = git("config", "user.name").stdout.strip()
git_email = git("config", "user.email").stdout.strip()

if not git_name or not git_email:
    raise ValueError(
        "Git author identity is missing. Configure user.name and user.email first."
    )

# 4. Commit
commit = git(
    "commit",
    "-m",
    "Add Cloudflare R2 deployment and cloud pipeline orchestration"
)

print("\nCOMMIT CREATED:")
print(commit.stdout)

# 5. Authenticate push without exposing token
with tempfile.TemporaryDirectory() as temp_dir:
    askpass = Path(temp_dir) / "git_askpass.sh"

    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding="utf-8"
    )
    askpass.chmod(0o700)

    env = os.environ.copy()
    env["GITHUB_TOKEN"] = github_token
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"

    push = git("push", "origin", "main", env=env)

    print("\nGITHUB PUSH OUTPUT:")
    print(push.stdout or "Push completed.")

# 6. Verify repository status
print("\nFINAL GIT STATUS:")
print(git("status", "--short", "--branch").stdout)

print("\nPHASE 7 CHANGES PUSHED TO GITHUB SUCCESSFULLY.")

GitHub token loaded securely.


CalledProcessError: Command '['git', 'diff', '--quiet']' returned non-zero exit status 1.